In [ ]:
import importlib, importlib.metadata as md, os, sys

# Colab에 깔린 pillow·requests 버전을 고정 (설치 중 덮어쓰기 방지)
open("/tmp/keep.txt", "w").write("\n".join(f"{p}=={md.version(p)}" for p in ["pillow", "requests"]))

%pip install -q -c /tmp/keep.txt beautifulsoup4 lxml pandas sentence-transformers faiss-cpu vllm transformers tokenizers huggingface-hub kiwipiepy bm25s nvidia-cutlass-dsl cuda-python
%pip -q install pyngrok fastapi uvicorn       # 라이브러리 설치
# 새로 깔린 cutlass 위치를 현재 세션에 등록 (재시작 없이 import 가능)
sys.path += [p for b in sys.path
             if os.path.isdir(p := os.path.join(b, "nvidia_cutlass_dsl", "dsl_packages"))][:1]
importlib.invalidate_caches()

In [ ]:
# CELL 1. 패키지 설치·import·공통 데이터 정의
import cutlass
import faiss
import hashlib
import json, re, html as htmlmod, requests
import os
import pandas as pd
import time
from bs4 import BeautifulSoup
from dataclasses import dataclass, field
from IPython.display import display, HTML
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from typing import Optional, Any, Literal
from urllib.parse import urlparse, parse_qsl, urlencode, urlsplit, urlunsplit
from google.colab import auth
from google.auth import default
from googleapiclient.discovery import build
from concurrent.futures import ThreadPoolExecutor
from difflib import SequenceMatcher
from html import escape
from collections import Counter          # 키워드 후보 출현 횟수 집계
from unicodedata import normalize        # 유니코드 정규화(전각/반각 통일)
from urllib.parse import parse_qs        # CELL 4의 _related_naver_url이 사용하는데 기존에 import가 빠져 있었음
from kiwipiepy import Kiwi               # 한국어 형태소 분석기
import threading, uvicorn
from fastapi import FastAPI , HTTPException
from fastapi.responses import HTMLResponse, JSONResponse, FileResponse
from pydantic import BaseModel
from google.colab import userdata
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Callable, Optional
import traceback
from __future__ import annotations
import numpy as np
from copy import deepcopy
import csv
import uuid, queue
from pyngrok import ngrok
# 드라이브 마운트
from google.colab import drive

drive.mount("/content/drive")

os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # JAX 선점 방지
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"        # TensorFlow 선점 방지
os.environ["JAX_PLATFORMS"] = "cpu"

SAVE_DIR = "/content/drive/MyDrive/VeriFive (kt 생성형 AI 팀 프로젝트 공유 문서함)/chunking_vectorDB"

@dataclass
class Article:
    url: Optional[str] = None                     # 기사 원문 주소
    title: Optional[str] = None                   # 기사 제목
    raw_body: str = ""                            # 정제 전 본문
    body: str = ""                                # 정제 후 본문
    published_at: Optional[str] = None            # 발행일
    press: Optional[str] = None                   # 언론사명
    clean_log: list = field(default_factory=list) # 정제 이력
    warnings: list = field(default_factory=list)  # 경고 모음


In [ ]:
# CELL 2. 헤더 및 언론사 매핑

# 봇차단을 막기위한 헤더 설정
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
}

# 네이버 기사 추출용
NAVER_BODY_SELECTOR = "#dic_area"


# NAVER API HUB에서 발급한 인증정보를 Colab 보안 비밀에 등록합니다.
# NAVER_CLIENT_ID: Client ID / NAVER_CLIENT_SECRET: Client Secret

NAVER_RELATED_API_URL = "https://naverapihub.apigw.ntruss.com/search/v1/news"


RELATED_NEWS_CONFIG = {
    "max_keywords": 6,               # 기사별 검색에 사용할 기존 키워드의 최대 개수
    "max_queries": 3,                # 입력 기사 한 건당 검색어 조합 수
    "results_per_query": 7,          # 검색어 한 개당 API 후보 수(1~100)
    "max_articles_per_question": 3,  # 입력 기사 한 건당 수집할 관련 기사 수
    "max_fetch_per_question": 6,     # 입력 기사 한 건당 본문 수집 시도 상한
    "min_body_chars": 200,           # 정제 후 본문 최소 글자 수
    "timeout": 10,                   # 검색 API 요청 제한 시간(초)
    "attempts": 3,                   # 일시적 API 오류의 최대 시도 횟수
    "request_interval": 0.1,         # 검색 API 요청 사이 대기 시간(초)
}


In [ ]:
# CELL 3. 데이터 정제 규칙

# 해야할 일 : 지속적으로 네이버 기사 투입하여 한번도 등장하지 않은 정제 규칙은 제외 검토

PHONE_PATTERN = r"(?<!\d)(?:01[016789]-?\d{3,4}-?\d{4}|0\d{1,2}-?\d{3,4}-?\d{4})(?!\d)"
RRN_PATTERN = r"(?<!\d)\d{6}-?[1-4]\d{6}(?!\d)"
EMAIL_PATTERN = r"[\w.%+-]+@[\w-]+(?:\.[\w-]+)+"
PII_PATTERNS = [("이메일 주소 삭제", EMAIL_PATTERN), ("전화번호 삭제", PHONE_PATTERN), ("주민등록번호 삭제", RRN_PATTERN)]

INLINE_NOISE = [
    ("인라인 사진 표지 삭제", r"<\s*(?:사진|영상|이미지)\s*>|\(\s*사진\s*\)|[·,]\s*사진(?=\))")
]
NOISE_LINE = [
    ("안내·링크 줄", r"^\s*[▶▷☞►■]\s*(?:https?://|www\.|관련\s*기사|기사\s*원문|원문\s*보기|카카오톡|이메일|홈페이지|바로가기|전화|메일|제보하기)"),
    ("사진·자료 출처 줄", r"^\s*[\[(]\s*(?:사진|영상|그래픽|자료|이미지)(?:\s+(?:출처|제공))?\s*[=:]|^\s*(?:사진|영상|그래픽|자료|이미지)\s+(?:출처|제공)\s*[=:]|^\s*(?:사진|영상|그래픽|자료|이미지)\s*=\s*[^\n]{1,40}$|^\s*(?:사진|그래픽|자료)\s*:\s*[^\n.]{1,40}$"),
    ("구독 유도 줄", r"^\s*[▷■▶]?\s*[^\n]{0,40}에서도\s+[^\n]{0,50}구독해\s*주세요!?\s*$"),
    ("기사 끝 표시", r"^\W*끝\W*$"),
    ("제보 안내 상용구", r"^\s*[▶▷☞►■]?\s*[^\n]{0,20}제보[^\n]{0,60}(?:시작입니다|기다립니다|기다리고\s*있습니다|받습니다|만들어집니다)[.!]?(?:\s*(?:홈페이지|이메일|e-?mail|메일|카카오톡|카톡|전화)\s*[=:：]\s*\S+)?\s*$"),
    ("연락처 라벨만 남은 줄", r"^\s*[▶▷☞►■]?\s*(?:이메일|e-?mail|메일|전화|카카오톡|카톡|홈페이지|연락처|제보)\s*[:：]?\s*$"),
    ("링크 전용 줄", r"^\s*[▶▷☞►■]?\s*(?:[가-힣A-Za-z ]{0,10}[:：]\s*)?(?:https?://|www\.)\S+\s*$"),
    ("대괄호 연락처 라벨만 남은 줄", r"^\s*[▶▷☞►■※]?\s*[\[(【]\s*(?:이메일|e-?mail|메일|전화|카카오톡|카톡|홈페이지|연락처|제보)\s*[\])】]\s*[:：]?\s*$"),
    ("대괄호 연락처 안내 줄", r"^\s*[▶▷☞►■※]?\s*[\[(【]\s*(?:이메일|e-?mail|메일|전화|카카오톡|카톡|홈페이지|연락처|제보)\s*[\])】][^\n]{0,40}(?:검색|채널\s*추가|친구\s*추가|바로가기)[^\n]{0,20}$"),
    ("제보 슬로건 줄", r"^\s*[※▶▷☞►■]\s*['\"“‘]?[^\n]{0,25}제보[^\n]{0,25}됩니다['\"”’]?[.!]?\s*$"),
    ("사진 설명 줄(끝에 출처 표기)", r"^\s*[\[【]?[^\n]{0,100}[(（]\s*(?:사진|영상|그래픽|자료|이미지)\s*[=:：][^\n()（）]{1,30}[)）]\s*[\]】]?\s*$"),
    ("자살예방 안내 문구", r"^\s*※[^\n]*(?:자살예방\s*상담전화|마들랜|129\.go\.kr)"),
]

JOURNALIST = r"(?:기자|특파원)"
INVISIBLE = re.compile(r"[\u200b-\u200f\u2028\u2029\u202a-\u202e\u2060\ufeff\x00-\x08\x0b\x0c\x0e-\x1f]")

In [ ]:
# CELL 4. 메타데이터 유틸 및 파싱 함수

# 해야할 일 : api 호출 방식 제거 여부 검토

# 언론사 추출
def _naver_press(soup):
    """네이버 기사 상단의 실제 언론사 로고를 읽어 포털명과 구분한다."""
    logo = soup.select_one(".media_end_head_top_logo img[alt]")
    if logo and logo.get("alt", "").strip(): return logo["alt"].strip()
    text_logo = soup.select_one(".media_end_head_top_logo_text")
    if text_logo and text_logo.get_text(" ", strip=True): return text_logo.get_text(" ", strip=True)
    return None

# 인터넷 기사 추출시 오류, 타임아웃 발생시 자동 재시도
def _fetch_html(url, timeout=5, attempts=2):
    for attempt in range(attempts):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            if r.encoding is None or r.encoding.lower() == "iso-8859-1": r.encoding = r.apparent_encoding
            return r.text
        except (requests.Timeout, requests.ConnectionError):
            if attempt + 1 >= attempts: raise
            time.sleep(1)

# HTML 헤더 태그 찾기
def _meta(soup, *names):
    for n in names:
        tag = soup.find("meta", attrs={"property": n}) or soup.find("meta", attrs={"name": n})
        if tag and tag.get("content"): return tag["content"].strip()


# ──────────────── 네이버 관련 기사 검색 유틸 ────────────────
# 에러 클래스, 텍스트·URL 정규화, API 인증 헤더, 검색 API 호출

class _RelatedNewsAPIError(RuntimeError):
    def __init__(self, message, fatal=False):
        super().__init__(message)
        self.fatal = fatal


def _related_normalize_text(value):
    return " ".join(normalize("NFKC", str(value or "")).casefold().split())


def _related_search_text(value):
    # 검색 API가 일치 단어에 붙인 강조 태그와 HTML 엔티티를 제거합니다.
    value = str(value or "").replace("<b>", "").replace("</b>", "")
    return " ".join(htmlmod.unescape(value).split())


def _related_naver_url(value):
    """네이버 일반 뉴스의 기사 식별자를 읽어 같은 기사 주소를 하나로 통일합니다."""
    try:
        parsed = urlparse(str(value or "").strip())
        if parsed.scheme not in ("http", "https") or parsed.username or parsed.password:
            return None
        if parsed.hostname not in ("n.news.naver.com", "news.naver.com"):
            return None
        parts = parsed.path.strip("/").split("/")
        if len(parts) == 3 and parts[0] == "article":
            oid, aid = parts[1:]
        elif len(parts) == 4 and parts[:2] == ["mnews", "article"]:
            oid, aid = parts[2:]
        elif parsed.path in ("/main/read.naver", "/main/ranking/read.naver", "/read.naver"):
            query = parse_qs(parsed.query)
            oid, aid = query.get("oid", [""])[0], query.get("aid", [""])[0]
        else:
            return None
        if not oid.isascii() or not aid.isascii() or not oid.isdigit() or not aid.isdigit():
            return None
        return f"https://n.news.naver.com/article/{oid}/{aid}"
    except ValueError:
        return None


def _related_api_headers():
    from google.colab import userdata
    try:
        client_id = userdata.get("NAVER_CLIENT_ID")
        client_secret = userdata.get("NAVER_CLIENT_SECRET")
    except Exception:
        raise RuntimeError(
            "Colab 보안 비밀에 NAVER API HUB의 NAVER_CLIENT_ID와 "
            "NAVER_CLIENT_SECRET을 등록하고 노트북 액세스를 허용하세요."
        ) from None
    if not client_id or not client_secret:
        raise ValueError("네이버 API 인증정보가 비어 있습니다.")
    return {"X-NCP-APIGW-API-KEY-ID": client_id, "X-NCP-APIGW-API-KEY": client_secret}


def _related_search_api(query, headers, config):
    params = {"query": query, "display": config["results_per_query"],
              "start": 1, "sort": "sim", "format": "json"}
    reason = "검색 API 요청 실패"
    for attempt in range(config["attempts"]):
        time.sleep(config["request_interval"])
        try:
            response = requests.get(NAVER_RELATED_API_URL, headers=headers, params=params,
                                    timeout=config["timeout"], allow_redirects=False)
        except (requests.Timeout, requests.ConnectionError):
            reason = "검색 API 연결 실패 또는 시간 초과"
        else:
            status = response.status_code
            if status == 200:
                try:
                    data = response.json()
                except ValueError:
                    raise _RelatedNewsAPIError("검색 API 응답이 올바른 JSON이 아닙니다.") from None
                if not isinstance(data, dict) or not isinstance(data.get("items"), list):
                    raise _RelatedNewsAPIError("검색 API 응답에 items 목록이 없습니다.")
                return data["items"]
            if status in (401, 403):
                raise _RelatedNewsAPIError(
                    f"네이버 API 인증 또는 권한 오류(HTTP {status}). "
                    "NAVER API HUB용 키와 뉴스 API 이용 설정을 확인하세요.", fatal=True)
            reason = f"검색 API HTTP {status}"
            if status != 429 and not 500 <= status < 600:
                raise _RelatedNewsAPIError(reason, fatal=True)
        if attempt + 1 < config["attempts"]:
            time.sleep(2 ** attempt)
    raise _RelatedNewsAPIError(f"{reason}: {config['attempts']}회 시도 후 중단")

In [ ]:
# CELL 5. 데이터 정제 함수

REQUIRED_TEXT_FIELDS = ("url", "title", "raw_body", "body")
BOILERPLATE_MARKERS = ("광고 문구", "광고 보기", "[광고]", "(광고)", "구독 신청", "구독하기", "구독 안내", "무단전재", "copyright")
SENT_SPLIT = re.compile(r"(?<!Mr\.)(?<!Mrs\.)(?<!Ms\.)(?<!Dr\.)(?<!Prof\.)(?<!U\.S\.)(?<![A-Z]\.)(?<!Inc\.)(?<!Co\.)(?<!Ltd\.)(?<!Corp\.)(?<!No\.)(?<!St\.)(?<!vs\.)(?<!^\d\.)(?<!^\d\d\.)(?<!\s\d\.)(?<!\s\d\d\.)(?<=\.)\s+")

def _head_patterns(press):
    """기사 머리말: (지역=언론사) / [언론사] / 'OOO 기자 =' — '=' 뒤가 언론사일 때만 머리말로 본다."""
    names = "|".join(map(re.escape, sorted({press} if press else set(), key=len, reverse=True))) or r"(?!)"  # press가 없으면 아무것도 매치하지 않음
    who = r"[가-힣]{2,5}(?:[\s,·]+[가-힣]{2,5}){0,2}"
    sep, bo, bc = r"[=:|ㅣ·/\-]", r"[(\[【<]", r"[)\]】>]"
    byline = rf"(?:{who}\s*(?:인턴\s+)?{JOURNALIST}\s*{sep}?\s*)"
    media_tail = rf"(?:{names}|[^\n)\]】>=]{{0,15}}(?:신문|일보|뉴스|방송|TV|경제|미디어|닷컴|투데이|타임스|저널|데일리))"
    return [
        ("(지역=언론사) 표기", rf"^\s*{bo}[^\n)\]】>=]{{1,30}}=\s*{media_tail}\s*{bc}\s*{byline}?"),
        ("[언론사] 표기",      rf"^\s*{bo}\s*(?:{names})(?:\s*(?:TV|단독|속보|종합|특파원|인터뷰))?\s*{bc}\s*{byline}?"),
        ("[매체명] 기자 표기", rf"^\s*{bo}[^\n)\]】>]{{1,20}}{bc}\s*{who}\s*(?:인턴\s+)?{JOURNALIST}\s*{sep}\s*"),
        ("기자 바이라인",      rf"^\s*(?:(?:{names})\s+)?{who}\s*(?:인턴\s+)?{JOURNALIST}\s*{sep}\s*")
    ]

def clean_news_body(text: str, meta: dict = None) -> tuple[str, list[dict]]:
    """구조 기반 정제: 문자 → 개인정보 → 짧은 노이즈 줄 → 기사 머리 → 기사 꼬리 → 중복.
    meta(title/press)에 이미 있는 정보는 본문에서 중복 제거한다."""
    meta = meta or {}
    press, title = meta.get("press"), meta.get("title")
    log = []
    def note(kind, detail, items):
        if items:
            log.append({"type": kind, "detail": detail if len(items) == 1 else f"{detail} ({len(items)}건)", "value": str(items)})

    # 1) HTML 엔티티, 보이지 않는 제어문자, 특수 공백과 인용 부호를 정리
    t = htmlmod.unescape(text)
    punctuation = re.findall(r"[．！？′″‥]", t)
    note("불필요한 문자 정제", "전각·인용 문장 부호를 표준 형태로 통일", punctuation)
    t = t.translate(str.maketrans({"．": ".", "！": "!", "？": "?", "′": "'", "″": '"', "‥": ".."}))
    junk = INVISIBLE.findall(t)
    note("불필요한 문자 정제", "보이지 않는 제어/깨짐 문자 삭제", [f"{len(junk)}자"] if junk else [])
    t = re.sub(r"[ \t\xa0\u3000]+", " ", INVISIBLE.sub("", t))

    # 2) 문장 속 노이즈(송고 일시 등) + 개인정보
    for kind, rules in (("노이즈 정제", INLINE_NOISE), ("개인정보 정제", PII_PATTERNS)):
        for detail, pat in rules:
            note(kind, detail, [m.group(0) for m in re.finditer(pat, t)])
            t = re.sub(pat, "", t)

    # 3) 짧은 노이즈 줄 삭제 (문단 단위)
    paras, dropped = [], {}
    for p in (x.strip() for x in t.split("\n")):
        hit = next((n for n, pat in NOISE_LINE if p and len(p) <= 150 and re.search(pat, p)), None)
        if hit:
            dropped.setdefault(hit, []).append(p)
        elif p:
            paras.append(p)
    for n, items in dropped.items():
        note("노이즈 정제", f"{n} 삭제", items)

    # 4) 머리·꼬리·중복 판정용으로 문장을 나누되 원문 문단 번호를 보존한다.
    #    (문장 단위로 NOISE_LINE을 다시 적용하지 않는다: 문단 중간의 본문 문장 오삭제 방지)
    lines, line_paragraphs = [], []
    for paragraph_index, paragraph in enumerate(paras):
        for sentence in (x.strip() for x in SENT_SPLIT.split(paragraph)):
            if sentence:
                lines.append(sentence)
                line_paragraphs.append(paragraph_index)

    # 5) 기사 머리: 처음 10줄 안의 (지역=언론사) · [언론사] · 'OOO 기자 =' (패턴별 1회)
    done = set()
    for i in range(min(10, len(lines))):
        for name, pat in _head_patterns(press):
            m = re.match(pat, lines[i])
            if m and name not in done:
                done.add(name)
                note("노이즈 정제", f"기사 머리 {name} 삭제", [m.group(0)])
                lines[i] = lines[i][m.end():].strip()

    if lines:
        filtered = [(line, pi) for line, pi in zip(lines, line_paragraphs) if line]
        if filtered:
            lines, line_paragraphs = zip(*filtered)
            lines, line_paragraphs = list(lines), list(line_paragraphs)
        else:
            lines, line_paragraphs = [], []

    # 6) 중복 제거(20자 이상 문장·제목만) 후, 같은 문단의 문장은 공백으로, 다른 문단은 원문처럼 줄바꿈 1번으로 연결한다.
    title_key = re.sub(r"\s+", "", title) if title else ""
    seen = {title_key} if title_key else set()
    out, out_paragraphs, dups = [], [], []
    for line, paragraph_index in zip(lines, line_paragraphs):
        key = re.sub(r"\s+", "", line)
        if key in seen and (len(key) >= 20 or key == title_key):
            dups.append(line)
            continue
        out.append(line)
        out_paragraphs.append(paragraph_index)
        if len(key) >= 20:
            seen.add(key)
    note("중복 정제", "반복 문장/제목 중복 삭제", dups)

    cleaned_parts = []
    for index, line in enumerate(out):
        if index:
            cleaned_parts.append("\n" if out_paragraphs[index] != out_paragraphs[index - 1] else " ")
        cleaned_parts.append(line)
    body = "".join(cleaned_parts).strip()
    kept_lines, leftover = [], []
    for line in body.split("\n"):
        (leftover if boilerplate_lines(line) else kept_lines).append(line)
    note("노이즈 정제", "잔여 광고·구독 문구 줄 삭제", [x.strip() for x in leftover])
    return "\n".join(kept_lines).strip(), log

def _apply_cleaning(art: Article, raw: str):
    """원본 보존 → 정제 (URL/텍스트 입력 공통)"""
    art.raw_body = raw
    art.body, art.clean_log = clean_news_body(raw, {'title': art.title, 'press': art.press})

# 공백 압축
def normalize_space(text: Any) -> str:
    return re.sub(r"\s+", " ", str(text or "")).strip()

# 잔여 광고 문구 탐지
def boilerplate_lines(text: str) -> list[str]:
    hits = []
    for line in (text or "").splitlines():
        stripped = line.strip()
        if stripped and len(stripped) <= 100 and any(marker in stripped.lower() for marker in BOILERPLATE_MARKERS):
            hits.append(stripped[:300])
    return hits

def validate_record_object(art: Article, index: int = 0) -> dict[str, Any]:
    errors, warnings = [], []
    missing = [field for field in REQUIRED_TEXT_FIELDS if not str(getattr(art, field, "")).strip()]
    if missing:
        errors.append("필수 값 누락: " + ", ".join(missing))
    if art.url and not art.url.startswith(("https://", "http://")):
        errors.append("기사 URL 형식이 올바르지 않습니다.")

    raw_chars, clean_chars = len(art.raw_body.strip()), len(art.body.strip())
    reduction_pct = round((1 - clean_chars / raw_chars) * 100, 2) if raw_chars else None

    if raw_chars and clean_chars and clean_chars / raw_chars < 0.25:
        warnings.append("정제 후 본문이 원문의 25% 미만입니다. 과다 삭제 여부를 확인하세요.")
    if not str(art.press or "").strip():
        warnings.append("언론사 메타데이터가 없습니다.")
    if not str(art.published_at or "").strip():
        warnings.append("작성일 메타데이터가 없습니다.")
    for w in art.warnings:
        if str(w).strip() not in warnings:
            warnings.append("팀 코드 경고: " + str(w).strip())

    return {
        "index": index,
        "url": art.url or "",
        "title": art.title or "",
        "press": art.press or "",
        "published_at": art.published_at or "",
        "raw_chars": raw_chars,
        "clean_chars": clean_chars,
        "reduction_pct": reduction_pct,
        "duplicate": False,
        "status": "오류" if errors else ("확인 필요" if warnings else "통과"),
        "errors": errors,
        "warnings": warnings,
        "issues": errors + warnings
    }

def validate_articles_object_pipeline(articles: list[Article]) -> list[dict[str, Any]]:
    results = [validate_record_object(art, index=i + 1) for i, art in enumerate(articles)]
    groups: dict[str, list[int]] = {}
    for i, art in enumerate(articles):
        normalized = normalize_space(art.body).lower()
        if normalized:
            digest = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
            groups.setdefault(digest, []).append(i)
    for indexes in groups.values():
        if len(indexes) > 1:
            for i in indexes:
                results[i]["duplicate"] = True
                warning = "동일 정제 본문 중복"
                results[i]["warnings"].append(warning)
                results[i]["issues"].append(warning)
                if results[i]["status"] == "통과":
                    results[i]["status"] = "확인 필요"
    return results

def before_after_table(results: list[dict[str, Any]]) -> pd.DataFrame:
    columns = ["index", "title", "press", "raw_chars", "clean_chars", "reduction_pct", "duplicate", "status", "issues", "url"]
    return pd.DataFrame(results)[columns]

In [ ]:
# CELL 6. 메인 실행부

def extract_from_url(url: str) -> Article:
    art = Article(url=url)
    try:
        page = _fetch_html(url)
    except Exception as e:
        art.warnings.append(f"페이지 가져오기 실패: {e}")
        return art
    soup = BeautifulSoup(page, "lxml")

    body = ""
    node = soup.select_one(NAVER_BODY_SELECTOR)
    if node:
        for related in node.find_all("div", recursive=False):
            if related.find("figure") and re.search(r"https?://", related.get_text(" ", strip=True)):
                related.decompose() # 본문 뒤에 붙은 관련 기사 카드 제거
        # 사진과 설명을 함께 감싼 span까지 제거해야 바깥 em 캡션이 남지 않는다.
        for photo in list(node.select(".end_photo_org")):
            wrapper = photo.parent
            if wrapper and wrapper.name == "span" and wrapper.find("em", recursive=False):
                wrapper.decompose()
            else:
                photo.decompose()
        for junk in node.select("script, style, .img_desc, figure"):
            junk.decompose()
        # b·i·span 등 인라인 서식 태그는 문단 구분이 아니므로 풀고 인접 문자열을 합친다.
        for inline in list(node.find_all(["b", "strong", "i", "u", "em", "span"])):
            inline.unwrap()
        node.smooth()
        body = node.get_text("\n", strip=True)
    else:
        art.warnings.append("본문 영역을 찾지 못했습니다: 네이버 페이지 구조 변경 또는 지원하지 않는 화면일 수 있습니다.")
        return art

    title = _meta(soup, "og:title", "twitter:title") or (soup.title.string if soup.title else None)
    art.press = _naver_press(soup)
    date_tag = soup.select_one(".media_end_head_info_datestamp_time")
    published_raw = date_tag.get("data-date-time") if date_tag else None
    art.published_at = str(published_raw).strip() or None if published_raw else None
    art.title = title

    _apply_cleaning(art, body)

    for ok, msg in [(art.title, "제목 추출 실패"), (art.published_at, "발행일 추출 실패: 수동 입력 필요 (상대 시점 해석 불가)"), (art.press, "언론사 추출 실패"), (len(art.body) >= 200, "본문이 매우 짧음: 추출 실패 또는 유료/로그인 기사 가능성")]:
        if not ok:
            art.warnings.append(msg)
    return art

In [ ]:
# CELL 7. 본문 키워드 추출

# Kiwi 품사 태그: NNG=일반명사, NNP=고유명사, SL=외국어(영문 등), SH=한자
_KEYWORD_NOUN_TAGS = frozenset(("NNG", "NNP", "SL", "SH"))


def _keyword_candidate_counts(text, analyzer):
    text = normalize("NFKC", text)        # 전각 문자 등을 통일해 같은 표현이 다르게 세어지는 것을 방지
    tokens = analyzer.tokenize(text)      # 형태소 분석
    counts = Counter()                    # {키워드 후보: 출현 횟수}
    noun_spans = []                       # 명사(또는 붙어 있는 명사 덩어리)의 (시작, 끝) 위치

    for index, token in enumerate(tokens):
        if token.tag in _KEYWORD_NOUN_TAGS:
            # 붙어 있는 명사 형태소는 한 덩어리로 합칩니다. (예: "가공"+"식품" → "가공식품")
            if noun_spans and noun_spans[-1][1] == token.start:
                noun_spans[-1] = (noun_spans[-1][0], token.end)
            else:
                noun_spans.append((token.start, token.end))
        # SN=숫자. 바로 뒤에 단위가 붙어 있으면 "2.8%", "3개월"처럼 하나의 후보로 만듭니다.
        elif token.tag == "SN" and index + 1 < len(tokens):
            unit = tokens[index + 1]
            if (
                unit.tag in ("NNB", "SW")
                and unit.start >= token.end
                and not text[token.end:unit.start].strip(" \t")
            ):
                counts[text[token.start:token.end] + text[unit.start:unit.end]] += 1

    for index, (start, end) in enumerate(noun_spans):
        counts[text[start:end]] += 1      # 명사 단독 후보
        # 뒤따르는 명사를 최대 2개까지 이어 붙여 2~3단어 명사구 후보를 만듭니다.
        for next_start, next_end in noun_spans[index + 1:index + 3]:
            if next_start < end or text[end:next_start].strip(" \t"):
                break
            counts[" ".join(text[start:next_end].split())] += 1
            end = next_end

    return counts

# 본문의 표현을 출현 횟수와 제목으로 정렬해 상위 top_k 개를 반환
def generate_chunk_keywords(keyword_args: dict):
    text = keyword_args["text"]
    title = keyword_args["title"]
    analyzer = keyword_args["analyzer"]
    top_k = keyword_args.get("top_k", 8)

    if top_k <= 0:
        return []
    candidates = _keyword_candidate_counts(text, analyzer)
    title_candidates = {term.casefold() for term in _keyword_candidate_counts(title or "", analyzer)}
    # 정렬 기준: ① 제목에 등장하는 표현 ② 본문 출현 횟수 ③ 표현 길이
    ranked = sorted(
        candidates,
        key=lambda term: (term.casefold() in title_candidates, candidates[term], len(term)),
        reverse=True,
    )
    keywords, seen = [], set()
    for term in ranked:
        key = term.casefold()
        if key not in seen:
            keywords.append(term)
            seen.add(key)
        if len(keywords) >= top_k:
            break
    return keywords

# 기사 단위 키워드 추출 청킹 단계 이전에 키워드를 추출 하기 때문에?

# [추가] 질문 상투어: 질문 틀에 붙는 일반 명사. 검색어로 쓰면 오히려 검색 품질을 떨어뜨려 후보에서 제외합니다.
QUESTION_STOPWORDS = frozenset((
    "사실", "여부", "진위", "내용", "이유", "관련", "정말", "진짜", "팩트체크", "팩트", "의미", "생각", "질문",
    "문제", "경우", "정도", "부분", "상황", "이야기", "얘기", "뉴스", "기사", "소식", "정보", "확인",
    "이번", "최근", "오늘", "어제", "올해", "요즘", "누구", "무엇", "어디", "언제", "어떻게", "왜",
    "영향", "관계", "때문", "원인",   # 질문 틀에 자주 붙는 말(검색 필터를 거의 무조건 통과시켜 근거 품질을 떨어뜨림)
))


def extract_question_keywords(question, analyzer, top_k=18):
    """[추가] 질문 한 문장에서 검색용 키워드 후보를 뽑습니다.
    본문용 정렬(①제목 포함 ②출현 횟수 ③길이)은 한 문장에서는 의미가 없어(제목=본문, 횟수는 대부분 1, 길이순이면 긴 명사구가
    핵심 단어보다 앞섬) 쓰지 않고, ①고유명사·숫자 표현 우선 ②질문에 나온 순서 ③같은 위치면 짧은(단일) 표현 우선으로 정렬합니다."""
    if top_k <= 0:
        return []
    text = normalize("NFKC", str(question or ""))
    candidates = _keyword_candidate_counts(text, analyzer)
    proper_forms = {token.form for token in analyzer.tokenize(text) if token.tag == "NNP"}   # 고유명사 표면형
    ranked = []
    for term in candidates:
        words = term.split()
        if any(word.casefold() in QUESTION_STOPWORDS for word in words):
            continue                                       # 질문 상투어가 들어간 표현 제외
        has_digit = any(ch.isdigit() for ch in term)
        if len(term) < 2 and not has_digit:
            continue                                       # 한 글자 표현 제외
        has_proper = any(form in term for form in proper_forms)
        position = text.find(words[0])
        ranked.append((not (has_digit or has_proper), position if position >= 0 else len(text), len(words), term))
    ranked.sort()
    keywords, seen = [], set()
    for *_, term in ranked:
        key = term.casefold()
        if key not in seen:
            keywords.append(term)
            seen.add(key)
        if len(keywords) >= top_k:
            break
    return keywords


In [ ]:
# CELL 8. 기사 단위 키워드 추출

#  검색어 조합·네이버 검색·근거 기사 수집

def _related_build_queries(keywords, max_queries):
    # 포함 관계인 표현을 중복 조합하지 않고, 추출된 명사구의 표기를 유지합니다.
    terms = []
    for keyword in keywords:
        key = _related_normalize_text(keyword)
        if key and not any(key in other or other in key for _, other in terms):
            terms.append((keyword, key))
    if not terms:
        return []
    words = [term for term, _ in terms]

    # 키워드 번호(0=A, 1=B, 2=C, 3=D)의 쌍. 쌍마다 겹치는 키워드가 달라서
    # 한 키워드가 질문과 어긋나도 다른 검색어가 보완합니다.
    patterns = [(0, 1), (2, 3), (0, 2), (1, 2), (1, 3)]
    combinations = [[words[i] for i in p] for p in patterns if max(p) < len(words)]
    if not combinations:                      # 키워드가 1개뿐이면 그대로 검색
        combinations = [words]
    return list(dict.fromkeys(" ".join(c) for c in combinations))[:max_queries]


def build_search_plan(question_text, keywords, config):
    """[추가] 질문 한 건의 키워드·검색어를 만듭니다."""
    # 순위 순으로 훑되, 이미 고른 표현과 포함 관계("물가"⊂"물가 상승")인 후보는 건너뛴 뒤 max_keywords개로 자릅니다.
    # (후보를 먼저 자르면 상위권이 포함 관계 표현으로만 채워져 검색어가 1개로 줄어듭니다.)
    selected_keywords, kept_keys = [], []
    for keyword in keywords:
        if not isinstance(keyword, str):
            raise TypeError("기사 키워드는 문자열 목록이어야 합니다.")
        keyword = " ".join(keyword.split())
        key = _related_normalize_text(keyword)
        if key and not any(key in other or other in key for other in kept_keys):
            kept_keys.append(key)
            selected_keywords.append(keyword)
        if len(selected_keywords) >= config["max_keywords"]:
            break
    return {
        "question_text": question_text,
        "keywords": selected_keywords, "queries": _related_build_queries(selected_keywords, config["max_queries"]),
    }


def collect_naver_related_articles(question_text, keywords, config, known_urls=None):
    """입력 기사를 변경하지 않고 네이버 관련 기사와 연결 정보를 반환합니다."""
    config = dict(config)
    for key in ("max_keywords", "max_queries", "results_per_query", "max_articles_per_question",
                "max_fetch_per_question", "min_body_chars", "attempts"):
        if type(config[key]) is not int or config[key] < 1:
            raise ValueError(f"{key}는 1 이상의 정수여야 합니다.")
    if config["results_per_query"] > 100:
        raise ValueError("results_per_query는 100 이하여야 합니다.")
    if config["timeout"] <= 0 or config["request_interval"] < 0:
        raise ValueError("시간 제한은 양수, 대기 시간은 0 이상이어야 합니다.")

    plan = build_search_plan(question_text, keywords, config)
    result = {"question": plan, "related_articles": [], "reused_urls": [],
            "links": [], "search_log": [], "collection_log": []}

    if not plan["queries"]:
        result["collection_log"].append({
            "status": "검색 생략", "reason": "사용할 키워드가 없습니다."})
        return result

    headers = _related_api_headers()
    known_urls = set(known_urls or [])
    seen_body_digests = set()          # 정제 본문이 같은 기사가 근거로 중복 채택되는 것을 막는 용도
    selected = result["related_articles"]
    candidates = {}

    for query in plan["queries"]:
        try:
            hits = _related_search_api(query, headers, config)
        except _RelatedNewsAPIError as exc:
            if exc.fatal:
                raise
            result["search_log"].append({"query": query,
                "status": "검색 실패", "count": 0, "reason": str(exc)})
            continue
        result["search_log"].append({"query": query,
            "status": "검색 완료", "count": len(hits), "reason": ""})
        for rank, hit in enumerate(hits, start=1):
            if not isinstance(hit, dict):
                continue
            # originallink는 출처 정보로만 보관하며 본문 수집에는 사용하지 않습니다.
            url = _related_naver_url(hit.get("link"))
            if not url:
                result["collection_log"].append({
                    "url": hit.get("link") or "", "status": "후보 제외",
                    "reason": "지원하는 네이버 뉴스 기사 URL이 아닙니다."})
                continue
            title = _related_search_text(hit.get("title"))
            description = _related_search_text(hit.get("description"))
            searchable = _related_normalize_text(title + " " + description)
            matched = [term for term in plan["keywords"]
            if _related_normalize_text(term) in searchable]
            candidate = candidates.setdefault(url, {"url": url, "title": title,
                "original_url": hit.get("originallink") or "", "queries": [],
                "matched_keywords": [], "best_rank": rank})
            candidate["best_rank"] = min(candidate["best_rank"], rank)
            if query not in candidate["queries"]:
                candidate["queries"].append(query)
            candidate["matched_keywords"] = list(dict.fromkeys(candidate["matched_keywords"] + matched))

    ranked = sorted(candidates.values(), key=lambda candidate: (
        -len(candidate["matched_keywords"]), candidate["best_rank"], -len(candidate["queries"])))

    def _prefetch(u):
        try:
            return extract_from_url(u)
        except Exception as e:
            return e
    to_fetch = [c["url"] for c in ranked
                if c["url"] not in known_urls][:config["max_articles_per_question"] + 3]
    with ThreadPoolExecutor(max(1, len(to_fetch))) as pool:
        prefetched = dict(zip(to_fetch, pool.map(_prefetch, to_fetch)))

    fetch_count = 0
    for candidate in ranked:
        if len(selected) + len(result["reused_urls"]) >= config["max_articles_per_question"]:
            break
        url = candidate["url"]
        if url in known_urls:                       # candidate_id 줄은 삭제
            result["reused_urls"].append(url)
            result["collection_log"].append({"url": url, "status": "기존 수집본 재사용", "reason": ""})
            result["links"].append({"question_text": question_text,
                "url": url, "title": candidate["title"], "press": None,
                "published_at": None, "body_chars": None, "original_url": candidate["original_url"],
                "queries": list(candidate["queries"]), "matched_keywords": list(candidate["matched_keywords"]),
                "best_search_rank": candidate["best_rank"], "warnings": []})
            continue
        if fetch_count >= config["max_fetch_per_question"]:
            break
        fetch_count += 1
        try:
            art = prefetched.get(url)
            if art is None or isinstance(art, Exception) or not art.body:
                art = extract_from_url(url)      # 미리 받기에 실패했으면 한 번 더 단독으로 시도
            reason = ""
            if len((art.body or "").strip()) < config["min_body_chars"]:
                reason = f"정제 본문이 {config['min_body_chars']}자 미만입니다."
                if art.warnings:
                    reason += " " + " / ".join(map(str, art.warnings))
            else:
                check = validate_record_object(art)
                digest = hashlib.sha256(normalize_space(art.body).lower().encode("utf-8")).hexdigest()
                if check["errors"]:
                    reason = "정제 검증 오류: " + " / ".join(check["errors"])
                elif digest in seen_body_digests:
                    reason = "이미 채택한 근거 기사와 정제 본문이 같습니다."
                else:
                    seen_body_digests.add(digest)
        except Exception as exc:
            art, reason = None, f"본문 수집 오류: {type(exc).__name__}"
        result["collection_log"].append({"url": url,
            "status": "수집 실패" if reason else "수집 완료", "reason": reason})
        if reason:
            continue
        selected.append(art)
        result["links"].append({"question_text": question_text,
            "url": url, "title": art.title or candidate["title"], "press": art.press,
            "published_at": art.published_at, "body_chars": len(art.body),
            "original_url": candidate["original_url"],
            "queries": list(candidate["queries"]),
            "matched_keywords": list(candidate["matched_keywords"]),
            "best_search_rank": candidate["best_rank"], "warnings": list(art.warnings)})
    return result

In [ ]:
# CELL 9. 실행·결과 확인

_related_required = ("extract_from_url", "RELATED_NEWS_CONFIG",
                     "extract_question_keywords", "collect_naver_related_articles")
_related_missing = [name for name in _related_required if name not in globals()]
if _related_missing:
    raise RuntimeError("CELL 1~9를 먼저 실행하세요. 없는 항목: " + ", ".join(_related_missing))

# 후보를 넉넉히(max_keywords의 3배) 뽑아 두면, 포함 관계 표현을 걸러낸 뒤에도 max_keywords개가 남습니다.
KEYWORD_CANDIDATES_PER_QUESTION = RELATED_NEWS_CONFIG["max_keywords"] * 3
keyword_analyzer = globals().get("keyword_analyzer") or Kiwi()   # 로딩이 무거우니 함수 밖에서 한 번만


def collect_articles(question, known_urls=None, keywords=None):
    question = (question or "").strip()
    if not question:
        raise ValueError("질문이 비어 있습니다.")
    if keywords is None:
        keywords = extract_question_keywords(
            question, keyword_analyzer, top_k=KEYWORD_CANDIDATES_PER_QUESTION)
    return collect_naver_related_articles(
        question_text=question, keywords=keywords,
        config=RELATED_NEWS_CONFIG, known_urls=known_urls)

QUESTION = ""    # 비워두면 실행하지 않습니다. 노트북에서 테스트할 때만 질문을 넣으세요.

if QUESTION.strip():
    naver_related_result = collect_articles(QUESTION)
    related_articles = naver_related_result["related_articles"]
    related_article_links = naver_related_result["links"]
    related_search_log = naver_related_result["search_log"]
    related_collection_log = naver_related_result["collection_log"]

    # [변경] 입력 기사별 → 질문 기준 출력으로 바꿨습니다.
    print("=== 질문의 검색 키워드·검색어 ===")
    _plan = naver_related_result["question"]
    display(pd.DataFrame([{
        "질문": _plan["question_text"],
        "키워드": ", ".join(_plan["keywords"]), "검색어": " / ".join(_plan["queries"]),
        "수집 건수": len(related_articles),
    }]))

    print(f"관련 기사 {len(related_articles)}건 수집 / 질문과의 연결 {len(related_article_links)}건")
    if related_article_links:
        display(pd.DataFrame(related_article_links)[[
            "question_text", "title", "press", "published_at", "body_chars", "matched_keywords", "url"
        ]])
    else:
        print("수집된 관련 기사가 없습니다. 검색·수집 이력을 확인하세요.")

    print("=== 검색 이력 ===")
    display(pd.DataFrame(related_search_log))
    print("=== 후보 제외·본문 수집 이력 ===")
    display(pd.DataFrame(related_collection_log))

    # 3. 근거 기사에도 입력 기사와 같은 품질 검증(필수값·과다 삭제·개인정보·광고 문구)을 적용
    # [변경] 위 설명의 '입력 기사' 검증은 삭제되어, 이제 근거 기사 검증만 남았습니다. (CELL 9에서 오류·중복 기사는 이미 채택 단계에서 제외됨)
    print("=== 근거 기사 정제 검증 ===")
    related_validation = validate_articles_object_pipeline(related_articles) if related_articles else []
    if related_validation:
        display(before_after_table(related_validation))
        print("상태별 건수:", pd.Series([r["status"] for r in related_validation]).value_counts().to_dict())
    else:
        print("검증할 근거 기사가 없습니다.")

    # 4. 청킹 단계로 넘기기 전 점검: 입력/근거 구분값과 연결 정보가 올바른지 확인
    # [변경] 근거 기사가 본문과 URL을 갖췄는지만 확인합니다.
    assert all(art.url and art.body for art in related_articles), "URL 또는 정제 본문이 없는 근거 기사가 있습니다."
    print(f"점검 완료: 근거 기사 {len(related_articles)}건")
else:
    print("QUESTION이 비어 있어 실행하지 않았습니다.")



In [ ]:
# CELL 10. 정제 회귀 테스트·실제 기사 수집 검증

articles = globals().get("related_articles", [])
results = globals().get("related_validation", [])
comparison = before_after_table(results) if results else pd.DataFrame()


# ──────────────── 전후 비교용 보조 함수 (이 셀 안에서만 사용) ────────────────

def _removed_spans(raw, clean):
    """정제 전·후 본문을 글자 단위(공백 제외)로 비교해, 정제 후에 남지 않은 원문 구간을 찾습니다.
    (정제 로그 문자열을 원문에서 검색하지 않으므로, 개인정보가 먼저 지워진 줄이나 공백이 바뀐 줄도 놓치지 않습니다.)"""
    raw_chars = [(i, ch) for i, ch in enumerate(raw) if not ch.isspace()]
    raw_text = "".join(ch for _, ch in raw_chars)
    clean_text = "".join(ch for ch in clean if not ch.isspace())
    kept = set()
    for block in SequenceMatcher(None, raw_text, clean_text, autojunk=False).get_matching_blocks():
        kept.update(range(block.a, block.a + block.size))
    spans, start, last = [], None, None
    for k, (pos, _) in enumerate(raw_chars):
        if k in kept:
            if start is not None:
                spans.append((start, last + 1))
                start = None
        else:
            if start is None:
                start = pos
            last = pos
    if start is not None:
        spans.append((start, last + 1))
    # 삭제 구간 사이에 공백·줄바꿈만 있으면 하나로 이어서 표시
    merged = []
    for st, en in spans:
        if merged and not raw[merged[-1][1]:st].strip():
            merged[-1] = (merged[-1][0], en)
        else:
            merged.append((st, en))
    return merged


def _highlight_removed(raw, spans):
    """삭제된 구간을 빨간 취소선으로 표시한 HTML을 만듭니다."""
    parts, pos = [], 0
    for start, end in spans:
        parts.append(escape(raw[pos:start]))
        parts.append(f'<mark style="background:#ffd6d6;color:#b00020;text-decoration:line-through">{escape(raw[start:end])}</mark>')
        pos = end
    parts.append(escape(raw[pos:]))
    return "".join(parts)


def show_before_after(art, result):
    """기사 1건의 정제 전·후 본문을 좌우 비교표로 보여줍니다. (빨간 취소선 = 삭제된 부분)"""
    box = "white-space:pre-wrap;word-break:break-word;max-height:420px;overflow:auto;font-size:13px;line-height:1.6;padding:8px;color:#222;background:#fff"
    th = "border:1px solid #bbb;padding:6px;background:#f0f0f0;color:#222;text-align:left"
    td = "border:1px solid #bbb;padding:0;vertical-align:top;color:#222;background:#fff"
    if not art.raw_body:
        display(HTML(f"<p><b>[{result['index']}] {escape(art.title or art.url or '')}</b> — 수집된 본문이 없어 비교할 수 없습니다. "
                     f"{escape(' / '.join(result['issues']))}</p>"))
        return
    removed = _removed_spans(art.raw_body, art.body)
    reduction = result["reduction_pct"]


##**청킹**
-> 추상화로 이전

##**임베딩**
-> 추상화로 이전

## **Google Drive 연결·저장**
-> 추상화로 이전

##**벡터 검색 기능**

In [ ]:
# CELL 16. 벡터 검색 기능
#
# 입력 딕셔너리(search_args):
#   query (str): 검색 질문
#   top_k (int): 상위 검색 결과 개수 (기본값: 5)
#   source_types (list[str], 선택): ★ 추가 — 이 출처 종류의 청크만 반환. 예: ["related"]. 없으면 전체
#   input_article_ids (list[str], 선택): ★ 추가 — "related" 청크 중 이 입력 기사에 연결된 것만 반환. 없으면 전체
#   candidate_k (int, 선택): ★ 추가 — 필터를 쓸 때 먼저 가져올 후보 수 (기본 50)
#
# 출력:
#   search_results (list[dict]): 유사도 내림차순 검색 결과
#
# 필요 객체:
#   model: 기존 BGE-M3 SentenceTransformer 모델
#   vector_store: {"index": FAISS 인덱스, "chunks": 청크 목록, ...}


# 질문을 BGE-M3로 벡터화하여 FAISS에서 유사한 청크를 검색하고 최소 점수로 필터링합니다.
# 입력: 질문, vector_store, 임베딩 모델 / 출력: 순위와 점수가 추가된 청크 딕셔너리 목록
def search_documents_advanced(search_args: dict) -> list[dict]:
    """
    개선된 벡터 검색 함수: 임계값(min_score) 필터링과 전역 변수 의존성 제거 적용
    """
    query = search_args["query"]
    vector_store = search_args["vector_store"]
    model = search_args["model"]
    top_k = search_args.get("top_k", 5)
    min_score = search_args.get("min_score", 0.2)
    source_types = search_args.get("source_types")            # ★ 추가: 예) ["input"] 또는 ["related"]. None 이면 필터 없음
    input_article_ids = search_args.get("input_article_ids")  # ★ 추가: 근거 청크를 입력 기사별로 거르는 필터. None 이면 필터 없음
    candidate_k = search_args.get("candidate_k", 50)          # ★ 추가: FAISS 는 사후 필터라 필터 전에 넉넉히 검색해야 합니다

    # 사용자 요청 top_k를 보정합니다. 저장된 벡터 수나 임계값에 따라 실제 반환 수는 줄 수 있습니다.
    # ★ 변경: 기존에는 3~5로 보정했지만(top_k = max(3, min(top_k, 5)) # 상위 데이터 최소 3개 에서 최대 5개 사이로 전달하게끔 만듬),
    #   입력 기사용(2개)·근거 기사용(4개)을 따로 검색하기 위해 1~10으로 넓혔습니다.
    top_k = max(1, min(top_k, 10))

    # 빈 질문이나 문자열이 아닌 입력을 사전에 차단합니다.
    if not isinstance(query, str) or not query.strip():
        raise ValueError("query는 비어 있지 않은 문자열이어야 합니다.")

    # vector_store에서 인덱스와 원문 청크 목록을 분리해 사용합니다.
    faiss_index = vector_store.get("index")
    chunks = vector_store.get("chunks")

    # 인덱스가 없거나 벡터가 0개이면 검색할 데이터가 없어 빈 목록을 반환합니다.
    if faiss_index is None or faiss_index.ntotal == 0:
        return []

    # 1. 질문 임베딩 및 float32 형변환
    # 검색 질문을 저장 문서와 동일한 임베딩 모델로 변환합니다.
    # FAISS 연산에 사용할 수 있도록 출력 배열을 float32로 맞춥니다.
    query_embedding = model.encode(
        [query.strip()],
        convert_to_numpy=True,
        show_progress_bar=False,
    ).astype("float32")

    # 2. L2 정규화 (코사인 유사도 검색 최적화)
    # 질문 벡터를 단위 벡터로 정규화합니다.
    # CELL10처럼 저장 벡터도 정규화한 IndexFlatIP일 때 내적 점수는 코사인 유사도입니다.
    faiss.normalize_L2(query_embedding)

    # 3. FAISS 상위 K개 검색
    # 데이터 개수보다 많은 결과를 요청하지 않도록 실제 FAISS 검색 개수 k를 정합니다.
    # ★ 변경: 필터(source_types / input_article_ids)를 쓰면 필터 후 결과가 0~1개로 줄지 않도록 candidate_k개를 먼저 가져옵니다.
    use_filter = bool(source_types) or bool(input_article_ids)
    k = min(max(candidate_k, top_k) if use_filter else top_k, faiss_index.ntotal)
    # scores는 유사도 점수, indices는 인덱스 내 벡터의 위치를 담습니다.
    scores, indices = faiss_index.search(query_embedding, k)

    # 4. 검색 결과 구성 및 임계값 필터링
    # 검색 순서에 따라 원문 청크와 점수를 결합할 결과 리스트를 생성합니다.
    search_results = []
    wanted_inputs = set(input_article_ids or [])   # ★ 추가: 허용할 입력 기사 ID 집합
    # 질문은 1개이므로 점수·인덱스 배열의 첫 번째 행([0])을 순회합니다.
    for rank, (score, chunk_idx) in enumerate(zip(scores[0], indices[0]), start=1):
        # FAISS가 유효한 인덱스를 반환하지 않은 항목은 건너뜁니다.
        if chunk_idx < 0:
            continue

        # 최소 유사도 점수 미만인 경우 제외 (노이즈 방지)
        # min_score 미만이면 제외합니다. 예: 0.3 미만 결과 제외.
        if float(score) < min_score:
            continue

        # 검색 인덱스의 위치를 이용해 동일 순서로 보관된 원문 청크를 가져옵니다.
        chunk = chunks[int(chunk_idx)]

        # ★ 추가: 출처 종류 필터 (예: 근거 기사 청크만)
        if source_types and chunk.get("source_type") not in source_types:
            continue
        # ★ 추가: 근거 청크는 지정한 입력 기사에 연결된 것만 남깁니다.
        #   (입력 청크는 question_ids 가 비어 있으므로 이 필터를 적용하지 않습니다.)
        if (wanted_inputs and chunk.get("source_type") == "related"
                and not wanted_inputs & set(chunk.get("question_ids") or [])):
            continue
        # 청크 원본 필드는 유지하고 통과 결과에 대해서만 rank·score를 추가합니다.
        search_results.append({
            **chunk,
            "rank": len(search_results) + 1,  # 유효 결과 기준 순위 재부여
            "score": float(score),
        })
        # ★ 추가: 필터 통과 결과가 top_k개가 되면 멈춥니다.
        if len(search_results) >= top_k:
            break

    # 최종적으로 임계값을 통과한 결과만 유사도 순서대로 반환합니다.
    return search_results


# ★ 추가: 검증 대상 기사(input)와 근거 기사(related)를 따로 검색해 하나의 목록으로 합칩니다.
# 흐름: ① 입력 기사 청크 top-k(검증 대상) → ② 그 청크의 기사 ID에 연결된 근거 청크 top-k(근거)
#       입력 기사 청크가 하나도 안 잡히면 근거 기사 전체에서 검색합니다.
# 반환: [입력 청크..., 근거 청크...] 순서의 list[dict]. 각 항목에 retrieval_group("input"/"related")이 붙습니다.
def retrieve_evidence(args: dict) -> list[dict]:
    base = {"query": args["query"], "vector_store": args["vector_store"], "model": args["model"]}

    # ① 검증 대상 기사 청크
    input_hits = search_documents_advanced({
        **base, "source_types": ["input"],
        "top_k": args.get("input_top_k", 2),
        "min_score": args.get("input_min_score", 0.3),
    })
    # 입력 청크가 속한 기사 ID들(중복 제거, 순서 유지)
    target_ids = list(dict.fromkeys(hit["article_id"] for hit in input_hits))

    # ② 그 입력 기사에 연결된 근거 기사 청크
    related_hits = search_documents_advanced({
        **base, "source_types": ["related"],
        "input_article_ids": target_ids or None,
        "top_k": args.get("related_top_k", 4),
        "min_score": args.get("related_min_score", 0.3),
    })

    for hit in input_hits:
        hit["retrieval_group"] = "input"
    for hit in related_hits:
        hit["retrieval_group"] = "related"
    return input_hits + related_hits

## 키워드 검색

BM25는 청크별로 생성한 키워드 목록을 검색합니다.
현재 청킹 결과를 검색하려면 `CELL 14-1`에서 `RUN_KEYWORD_SEARCH`를 켭니다.
새 런타임에서 저장본을 검색하려면 `CELL 1 → 8 → 11 → 14-1 → 14-2` 순서로 실행하고, `CELL 14-1`의 두 검색 설정을 켭니다.
인덱스를 준비한 뒤에는 `CELL 14-2`에서 검색어를 바꿔 실행합니다.

In [ ]:
# !pip install bm25s

In [ ]:
# # ==============================================================================
# # BM25 키워드 검색 함수 및 인덱스 준비

# # [이 셀의 역할] 벡터 검색(의미 기반)과 별개로, 키워드 일치 기반 검색(BM25)을 준비합니다.
# #   - 벡터 검색: "비슷한 의미"를 찾음 → 표현이 달라도 찾지만, 고유명사·용어("반대매매") 정확 일치에는 약할 수 있음
# #   - BM25 검색: "같은 단어"를 찾음 → 정확한 용어 일치에 강함
# # [주의] BM25는 청크 전체 본문이 아니라 Cell 9에서 청크마다 뽑은 키워드(4개)만 대상으로 검색합니다.

# import bm25s    # BM25 검색 라이브러리 (Cell 1에서 pip 설치)


# def _keyword_search_terms(text, analyzer):
#     """키워드와 검색어에 같은 형태소 분석과 표기 정규화를 적용합니다."""
#     # 저장 키워드와 사용자 검색어를 "같은 규칙"으로 쪼개야 서로 일치 여부를 정확히 비교할 수 있습니다.
#     text = normalize("NFKC", text).casefold()   # 전각→반각 등 표기 통일 + 소문자화 (대소문자 구분 없이 매칭)
#     tokens = analyzer.tokenize(text)            # 형태소 분석
#     terms = []                                  # 검색에 사용할 단어 목록
#     for position, token in enumerate(tokens):
#         # 명사(N*), 용언(V*: 동사·형용사), 관형사·부사(M*), 어근(XR), 외국어(SL), 한자(SH), 숫자(SN)만 남기고
#         # 조사·어미·문장부호 등은 버립니다. (Cell 8의 키워드 추출보다 넓은 범위: 사용자 검색어에는 동사 등도 섞이기 때문)
#         if token.tag.startswith(("N", "V", "M")) or token.tag in ("XR", "SL", "SH", "SN"):
#             terms.append(token.form.casefold())
#         # 숫자 바로 뒤에 단위(NNB 의존명사, SW 기호)가 붙어 있으면 "3개월", "2.8%"처럼 합친 단어도 추가합니다.
#         if token.tag == "SN" and position + 1 < len(tokens):
#             unit = tokens[position + 1]
#             if (
#                 unit.tag in ("NNB", "SW")
#                 and unit.start >= token.end
#                 and not text[token.end:unit.start].strip(" \t")   # 숫자와 단위 사이에 공백/탭 외의 글자가 없을 것
#             ):
#                 terms.append(token.form + unit.form.casefold())
#     return terms


# def build_keyword_search_store(keyword_store_args: dict):
#     """저장 키워드를 청크 ID로 연결해 메모리에 BM25 인덱스를 만듭니다."""
#     # ── 입력값 꺼내기 ──
#     keyword_chunks = keyword_store_args["keyword_chunks"]          # 청크 리스트 (검색 결과로 돌려줄 원문)
#     keywords_by_chunk = keyword_store_args["keywords_by_chunk"]    # {chunk_id: [키워드, ...]}
#     analyzer = keyword_store_args["analyzer"]                      # Kiwi 형태소 분석기

#     # 청크 ID가 중복되거나, 청크 목록과 키워드 목록의 ID 집합이 다르면 서로 다른 저장본이 섞인 것이므로 중단합니다.
#     chunk_ids = [chunk["chunk_id"] for chunk in keyword_chunks]
#     if len(set(chunk_ids)) != len(chunk_ids) or set(chunk_ids) != keywords_by_chunk.keys():
#         raise ValueError("청크와 키워드의 ID가 일치하지 않습니다. 같은 저장본을 사용하세요.")

#     # 서로 겹치는 대표 키워드에서 같은 검색어가 반복 계산되는 것을 방지합니다.
#     # 청크마다 "검색용 단어 목록" 하나(=BM25의 문서 1개)를 만듭니다. 순서는 chunk_ids(=청크 순서)와 같습니다.
#     # 예: 키워드 ["체감 물가", "물가"] → ["체감", "물가", "물가"] → dict.fromkeys로 중복 제거 → ["체감", "물가"]
#     documents = [
#         list(dict.fromkeys(                       # dict.fromkeys: 순서를 유지하면서 중복 제거
#             term
#             for keyword in keywords_by_chunk[chunk_id]
#             for term in _keyword_search_terms(keyword, analyzer)
#         ))
#         for chunk_id in chunk_ids
#     ]
#     keyword_index = None                          # 모든 청크의 단어 목록이 비어 있으면 인덱스를 만들 수 없어 None으로 둡니다.
#     if any(documents):
#         keyword_index = bm25s.BM25(method="lucene")            # Lucene 방식의 BM25 점수 계산식 사용
#         keyword_index.index(documents, show_progress=False)    # 단어 목록들로 BM25 인덱스 생성

#     # 검색 함수가 사용할 재료를 한 묶음으로 반환합니다. (벡터 검색의 vector_store와 비슷한 역할)
#     return {
#         "index": keyword_index,                   # BM25 인덱스 (documents 순서 = 청크 순서)
#         "chunks": keyword_chunks,                 # 결과로 돌려줄 원문 청크
#         "keywords_by_chunk": keywords_by_chunk,   # 일치 키워드를 표시할 때 사용
#         "analyzer": analyzer,                     # 검색어를 같은 방식으로 분석하기 위해 보관
#     }


# def search_chunks_by_keyword(keyword_search_args: dict) -> list[dict]:
#     """질의에 일치하는 키워드를 가진 청크를 BM25 점수 순으로 반환합니다."""
#     # ── 입력값 꺼내기 ──
#     keyword_query = keyword_search_args["keyword_query"]               # 검색어
#     keyword_search_store = keyword_search_args["keyword_search_store"] # build_keyword_search_store의 반환값
#     top_k = keyword_search_args.get("top_k", 5)                        # 가져올 상위 결과 수 (기본 5)

#     # ── 입력값 검증 ──
#     if not isinstance(keyword_query, str) or not keyword_query.strip():
#         raise ValueError("키워드 검색어를 입력하세요.")
#     if top_k < 1:
#         raise ValueError("top_k는 1 이상이어야 합니다.")

#     # 검색어도 저장 키워드와 같은 방식으로 쪼개고 중복을 제거합니다.
#     analyzer = keyword_search_store["analyzer"]
#     query_terms = list(dict.fromkeys(_keyword_search_terms(keyword_query, analyzer)))
#     keyword_index = keyword_search_store["index"]
#     # 인덱스가 없거나 검색어에서 쓸 단어가 하나도 안 나오면 빈 결과를 반환합니다.
#     if keyword_index is None or not query_terms:
#         return []

#     keyword_chunks = keyword_search_store["chunks"]
#     # BM25 검색. 질문이 1개이므로 [query_terms]로 감싸 전달하고, 결과의 첫 행([0])만 사용합니다.
#     # k는 전체 청크 수를 넘지 않도록 제한합니다. positions = 청크 위치 번호, scores = BM25 점수
#     positions, scores = keyword_index.retrieve(
#         [query_terms], k=min(top_k, len(keyword_chunks)), show_progress=False
#     )
#     query_term_set = set(query_terms)   # 일치 키워드 판별을 빠르게 하기 위한 집합
#     keyword_matches = []
#     for position, score in zip(positions[0], scores[0]):
#         # 점수가 0 이하 = 검색어와 겹치는 단어가 없는 청크 → 결과에서 제외합니다.
#         if score <= 0:
#             continue
#         chunk = keyword_chunks[int(position)]                                       # 위치 번호로 원문 청크 찾기
#         keywords = keyword_search_store["keywords_by_chunk"][chunk["chunk_id"]]     # 이 청크의 저장 키워드
#         keyword_matches.append({
#             **chunk,                                   # 청크의 기존 필드를 그대로 유지
#             "rank": len(keyword_matches) + 1,          # 통과한 결과 기준으로 순위 재부여
#             "score": float(score),                     # BM25 점수 (벡터 검색의 코사인 유사도와는 척도가 다름)
#             # 이 청크의 키워드 중 검색어와 겹치는 단어가 하나라도 있는 키워드만 골라 "왜 검색됐는지" 보여줍니다.
#             "matched_keywords": [
#                 keyword for keyword in keywords
#                 if query_term_set.intersection(_keyword_search_terms(keyword, analyzer))
#             ],
#         })
#     return keyword_matches


# # ── 실행 스위치 ──
# RUN_KEYWORD_SEARCH = False      # True이면 BM25 인덱스를 준비합니다.
# KEYWORD_SEARCH_FROM_FILES = False  # True이면 청크·키워드 JSON을 사용합니다.
# # (False/False: 아무것도 하지 않음 / True/False: 메모리의 현재 청킹 결과 사용 / True/True: Drive에 저장된 파일 사용)

# if RUN_KEYWORD_SEARCH:
#     if KEYWORD_SEARCH_FROM_FILES:
#         # [저장본 사용] 새 런타임에서 Cell 9를 다시 실행하지 않고 저장된 파일로 키워드 검색을 하는 경우입니다.
#         # 마운트된 Drive의 SAVE_DIR에서 파일을 직접 확인합니다.
#         if not (os.path.isfile(chunks_filepath) and os.path.isfile(keywords_filepath)):
#             raise FileNotFoundError(
#                 f"마운트된 Drive 폴더에 청크 또는 키워드 파일이 없습니다: {SAVE_DIR}"
#             )
#         # 키워드 JSON이 없으면(Cell 12에서 키워드를 저장하지 않은 경우) 안내 메시지와 함께 중단합니다.
#         if not os.path.isfile(keywords_filepath):
#             raise FileNotFoundError("키워드 JSON이 없습니다. CELL 9와 CELL 12에서 키워드를 생성하고 저장하세요.")
#         with open(chunks_filepath, "r", encoding="utf-8") as file:
#             keyword_chunks = json.load(file)            # 저장해 둔 청크 리스트
#         with open(keywords_filepath, "r", encoding="utf-8") as file:
#             keyword_search_data = json.load(file)       # 저장해 둔 {chunk_id: [키워드]}
#         keyword_search_analyzer = Kiwi()                # 새 런타임에는 분석기가 없으므로 새로 만듭니다.
#     else:
#         # [현재 실행 결과 사용] Cell 9에서 방금 만든 청크·키워드·분석기를 그대로 재사용합니다.
#         keyword_chunks = chunks
#         keyword_search_data = keywords_by_chunk
#         keyword_search_analyzer = keyword_analyzer

#     # 위에서 정한 재료로 BM25 인덱스를 만들어 keyword_search_store에 저장합니다. (Cell 14-2, 14-3에서 사용)
#     keyword_search_store = build_keyword_search_store({
#         "keyword_chunks": keyword_chunks,
#         "keywords_by_chunk": keyword_search_data,
#         "analyzer": keyword_search_analyzer,
#     })

In [ ]:
# # ==============================================================================
# # CELL 14-2. 키워드 검색 실행 및 결과 확인

# # [이 셀의 역할] Cell 14-1에서 만든 BM25 인덱스로 키워드 검색을 실행하고 결과를 출력합니다.
# #                검색어만 바꿔서 반복 실행할 수 있습니다.

# if RUN_KEYWORD_SEARCH:    # 14-1에서 인덱스를 준비하지 않았다면(False) 이 셀은 아무것도 하지 않습니다.
#     keyword_query = "반대매매"    # 키워드 검색어 (벡터 검색과 달리 문장보다 핵심 단어가 적합)
#     keyword_top_k = 4             # 가져올 상위 결과 수
#     keyword_search_results = search_chunks_by_keyword({
#         "keyword_query": keyword_query,
#         "keyword_search_store": keyword_search_store,   # Cell 14-1에서 만든 BM25 묶음
#         "top_k": keyword_top_k,
#     })

#     print(f"키워드 검색어: {keyword_query}")
#     print(f"검색 결과: {len(keyword_search_results)}개")   # 검색어와 겹치는 키워드가 없는 청크는 빠지므로 top_k보다 적을 수 있음
#     for keyword_result in keyword_search_results:
#         keyword_metadata = keyword_result["metadata"]       # 기사 제목·URL 등 출처 정보
#         print("=" * 60)
#         print(f"순위: {keyword_result['rank']} | BM25: {keyword_result['score']:.4f}")   # BM25 점수 (클수록 검색어와 잘 맞음)
#         print(f"일치 키워드: {', '.join(keyword_result['matched_keywords'])}")          # 검색어와 겹친 저장 키워드
#         print(f"청크 ID: {keyword_result['chunk_id']}")
#         print(f"기사 제목: {keyword_metadata.get('title')}")
#         print(f"기사 URL: {keyword_metadata.get('url')}")
#         print(f"청크 본문:\n{keyword_result['text']}")

In [ ]:
# # ==============================================================================
# # CELL 14-3. 하이브리드 검색 함수
# # ==============================================================================
# # [이 셀의 역할] 벡터 검색과 BM25 키워드 검색 결과를 하나로 합칩니다. (함수 정의만 하며 이 셀에서는 실행하지 않습니다.)
# # [합치는 방식: RRF(Reciprocal Rank Fusion)]
# #   두 검색의 점수는 척도가 달라(코사인 유사도 vs BM25) 직접 더할 수 없으므로, "순위"만 사용합니다.
# #   청크의 점수 = Σ 1 / (rrf_rank_constant + 각 검색에서의 순위)
# #   → 두 검색 모두에서 상위에 오른 청크가 가장 높은 점수를 받습니다.

# def search_hybrid_chunks(hybrid_search_args: dict) -> list[dict]:
#     """벡터 검색과 BM25 검색 결과를 청크 ID 기준으로 합쳐 RRF 순위를 반환합니다."""
#     # ── 입력값 꺼내기 (괄호 안 숫자는 지정하지 않았을 때의 기본값) ──
#     query = hybrid_search_args["query"]                                   # 검색 질문 (벡터·키워드 검색에 같은 문자열 사용)
#     vector_store = hybrid_search_args["vector_store"]                     # FAISS 인덱스 + 청크
#     model = hybrid_search_args["model"]                                   # BGE-M3 임베딩 모델
#     keyword_search_store = hybrid_search_args["keyword_search_store"]     # BM25 인덱스 + 청크
#     top_k = hybrid_search_args.get("top_k", 4)                            # 최종 반환 개수 (4)
#     candidate_k = hybrid_search_args.get("candidate_k", 10)               # 각 검색에서 합치기 전에 가져올 후보 수 (10)
#     min_score = hybrid_search_args.get("min_score", 0.3)                  # 벡터 검색 최소 유사도 (0.3) ※ RRF 점수에는 적용되지 않음
#     rrf_rank_constant = hybrid_search_args.get("rrf_rank_constant", 60)   # RRF 상수 (60). 클수록 순위 차이의 영향이 작아짐

#     # ── 입력값 검증 ──
#     if top_k < 1 or candidate_k < top_k or rrf_rank_constant < 1:
#         raise ValueError("top_k와 rrf_rank_constant는 1 이상, candidate_k는 top_k 이상이어야 합니다.")
#     # 두 검색이 서로 다른 청크 목록을 쓰면 chunk_id가 어긋나므로, 청크 목록이 완전히 같은지 확인합니다.
#     if vector_store["chunks"] != keyword_search_store["chunks"]:
#         raise ValueError("벡터 검색과 키워드 검색에 같은 청크 저장본을 사용하세요.")

#     # ── 1단계: 두 검색을 각각 실행해 후보를 모읍니다 ──
#     # ※ 벡터 검색 함수(Cell 13)는 top_k를 내부에서 3~5로 보정하므로, candidate_k=10을 줘도 벡터 후보는 최대 5개입니다.
#     vector_results = search_documents_advanced({
#         "query": query,
#         "vector_store": vector_store,
#         "model": model,
#         "top_k": candidate_k,
#         "min_score": min_score,
#     })
#     keyword_results = search_chunks_by_keyword({
#         "keyword_query": query,
#         "keyword_search_store": keyword_search_store,
#         "top_k": candidate_k,
#     })

#     # ── 2단계: 청크 ID 기준으로 합치며 RRF 점수를 누적합니다 ──
#     combined = {}    # {chunk_id: 합쳐진 결과 딕셔너리}
#     for channel, candidates in (("vector", vector_results), ("keyword", keyword_results)):
#         for hit in candidates:
#             # 처음 등장한 청크면 초기 항목을 만들고, 이미 있으면(두 검색 모두에 걸림) 기존 항목을 그대로 가져옵니다.
#             entry = combined.setdefault(hit["chunk_id"], {
#                 **hit,                      # 청크 원본 필드 유지
#                 "score": 0.0,               # 이후 RRF 점수로 사용할 누적값 (검색별 원래 점수로 덮어쓰이지 않도록 0으로 초기화)
#                 "vector_rank": None,        # 벡터 검색에서의 순위 (안 걸렸으면 None)
#                 "vector_score": None,       # 벡터 검색 유사도
#                 "keyword_rank": None,       # 키워드 검색에서의 순위 (안 걸렸으면 None)
#                 "keyword_score": None,      # BM25 점수
#                 "matched_keywords": [],     # 키워드 검색에서 일치한 키워드
#             })
#             entry[f"{channel}_rank"] = hit["rank"]       # vector_rank 또는 keyword_rank 기록
#             entry[f"{channel}_score"] = hit["score"]     # vector_score 또는 keyword_score 기록
#             entry["score"] += 1.0 / (rrf_rank_constant + hit["rank"])   # RRF 점수 누적: 순위가 높을수록(숫자가 작을수록) 더 크게 더해짐
#             if channel == "keyword":
#                 entry["matched_keywords"] = hit["matched_keywords"]     # 키워드 검색에서만 의미 있는 값

#     # ── 3단계: RRF 점수 내림차순으로 정렬하고 상위 top_k개를 순위와 함께 반환합니다 ──
#     # 점수가 같으면 chunk_id 오름차순으로 정렬해, 실행할 때마다 순서가 바뀌지 않게(결과가 항상 동일하게) 합니다.
#     ranked_results = sorted(combined.values(), key=lambda hit: (-hit["score"], hit["chunk_id"]))
#     return [
#         {**hit, "rank": rank}   # 합친 뒤의 최종 순위로 rank를 덮어씀. 이때 score는 코사인/BM25가 아니라 RRF 점수
#         for rank, hit in enumerate(ranked_results[:top_k], start=1)
#     ]

## RAG 답변 생성

위 벡터 검색(Cell 13)의 `query`와 `search_results`를 받아 **검색 결과 전체를 한 프롬프트로 구성하고 14B를 1회만 호출**해 답변을 만듭니다.

- 같은 기사의 청크는 같은 `[근거 N]`을 공유하고, 출처 URL은 기사 단위로 병합합니다.
- 자동 검사는 인용 형식·번호 범위·인용 근거의 숫자 포함 관계로 제한하며, 검사에 실패해도 답변을 임의로 고치지 않고 실패 이유를 그대로 보고합니다.
- 검색 결과가 비어 있으면 모델을 호출하지 않고 근거 부족 응답을 즉시 반환합니다.

In [ ]:
# CELL 18. 답변 생성 설정·시스템 프롬프트·검사용 정규식

"""검색 결과를 받아 근거 기반 답변과 비교용 일반 답변을 생성하는 모듈."""

# 한국어 생성 품질과 Colab GPU 메모리 요구를 함께 충족하기 위해 14B AWQ 모델을 사용한다.
MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct-AWQ"
# 모델과 토크나이저 revision을 고정해야 평가 중 모델 업데이트로 결과가 바뀌지 않는다.
MODEL_REVISION = "539535859b135b0244c91f3e59816150c8056698"
MAX_MODEL_LEN = 8192
MAX_NEW_TOKENS = 512
INSUFFICIENT = "근거 부족: 제공된 문서만으로는 답할 수 없습니다."
# ★ 추가: 인용([근거 N])을 허용할 기사 종류.
#   ("related",)          : 근거 기사만 인용 가능 (권장 — 입력 기사가 자기 주장을 스스로 뒷받침하는 순환 방지)
#   ("related", "input")  : 입력 기사도 인용 허용 (근거 기사가 부족할 때 임시로 사용)
CITABLE_SOURCE_TYPES = ("related",)
SYSTEM_PROMPT = """당신은 제공된 경제 뉴스 근거만으로 답하는 한국어 생성형 챗봇입니다.
질문에 직접 답하는 1~4개의 완결된 문장만 작성하되 필요한 문장만 쓰고 문장 수를 채우지 마세요.
'[검증 대상 기사 · 인용 불가]'로 표시된 기사는 어떤 주장을 확인해야 하는지 파악하는 용도로만 읽으세요. 이 기사의 내용을 [근거 N]으로 인용하거나 근거로 삼지 마세요.
검증 대상 기사의 주장은 '[근거 N]' 기사에 있는 내용만으로 판단하세요. 근거 기사가 그 주장을 뒷받침하는지, 다르게 보도했는지, 언급하지 않는지를 구분해서 쓰세요.
질문의 대상·기간·사건과 직접 관련된 근거만 사용하고, 검색 결과에 섞인 다른 주제의 기사는 제외하세요.
모든 근거를 억지로 사용할 필요는 없으며, 질문과 무관한 내용을 추가해 답변을 늘이지 마세요.
사건의 영향이나 원인을 묻는 경우 근거에 명시된 인과관계만 연결하고, 동시에 보도된 다른 상황이나 배경 지표를 그 사건의 영향으로 제시하지 마세요.
각 문장은 새 줄로 쓰고, 문장 끝에 직접 지지하는 출처를 [근거 N] 형식으로 붙이세요.
같은 기사에서 온 여러 청크는 같은 [근거 N]을 공유합니다.
근거를 자연스럽게 paraphrase하되 사실·수치·기간·조건·대상·분석 주체·한계를 바꾸지 마세요.
특정 조치나 제도를 설명할 때는 첫 문장에 조치·대상·목적을 쓰고, 다음 문장에 근거에 있는 시행일·적용 기간·비율·금액·한도를 별도로 모두 명시하세요.
괄호 안의 수치 조건을 포함해 질문과 관련된 수치가 한 근거에만 있어도 빠뜨리지 말고 서로 다른 근거의 세부 조건을 종합하세요.
날짜·기간·비율·금액·한도는 추론하거나 확장하지 말고 근거 원문 표현을 그대로 유지하며, 생략된 연도나 월을 보충하지 마세요.
답변의 핵심 조건과 수치가 근거에 있으면 빠뜨리지 말고 구체적으로 포함하세요.
가능성과 전망을 확정 사실로 바꾸거나 근거에 없는 인과를 추가하지 마세요.
제목·목록·서론·설명은 출력하지 마세요.
[근거 N] 기사만으로 질문에 직접 답할 수 없으면 정확히 '근거 부족: 제공된 문서만으로는 답할 수 없습니다.'라고만 답하세요."""
# 비교 조건에서는 모델과 디코딩 설정만 같게 하고 문서 근거와 RAG 지시는 제공하지 않는다.
PLAIN_SYSTEM_PROMPT = "당신은 한국어로 질문에 간결하고 명확하게 답하는 생성형 챗봇입니다."
CITATION_BLOCK = re.compile(r"(?:\[근거 [1-9]\d*\])(?:\s*\[근거 [1-9]\d*\])*$")
CITATION = re.compile(r"\[근거 ([1-9]\d*)\]")
CITATION_LIKE = re.compile(r"\[근거[^\]\n]*\]")
NUMBER = re.compile(r"\d+(?:[.,]\d+)*")

In [ ]:
# CELL 19. 검색 결과 정규화·기사 단위 출처 병합·프롬프트 구성

def normalize_url(value):
    """추적 매개변수 차이로 같은 기사가 여러 출처로 나뉘지 않게 URL을 정규화한다."""
    if not isinstance(value, str) or not value.strip():
        return ""
    parts = urlsplit(value.strip())
    query = urlencode([
        (key, val) for key, val in parse_qsl(parts.query, keep_blank_values=True)
        if not key.lower().startswith("utm_") and key.lower() not in {"fbclid", "gclid", "spm"}
    ])
    return urlunsplit((parts.scheme, parts.netloc, parts.path, query, ""))


def normalize_chunks(chunks):
    """팀 검색 dict와 Document 입력을 같은 형태로 바꾸고 빈 근거를 조기에 거부한다."""
    if not isinstance(chunks, (list, tuple)):
        raise TypeError("검색 결과는 dict 또는 Document의 list/tuple이어야 합니다.")
    normalized = []
    for index, item in enumerate(chunks, 1):
        if hasattr(item, "page_content"):
            item = {"text": item.page_content, "metadata": getattr(item, "metadata", {})}
        if not isinstance(item, dict):
            raise TypeError(f"검색 결과 {index}은 dict 또는 Document여야 합니다.")
        # 호출자의 검색 결과와 metadata를 변경하지 않도록 얕은 복사본을 사용한다.
        chunk = dict(item)
        chunk["text"] = chunk.get("text", chunk.get("page_content", ""))
        chunk["metadata"] = dict(chunk.get("metadata") or {})
        if not isinstance(chunk["text"], str) or not chunk["text"].strip():
            raise ValueError(f"검색 결과 {index}의 본문이 비어 있습니다.")
        normalized.append(chunk)
    return normalized


def group_sources(chunks, citable_types=None):
    """청크 순서를 유지하면서 URL(없으면 article_id) 기준으로 기사 번호를 병합한다.

    ★ 변경: source_type 이 citable_types(기본 CITABLE_SOURCE_TYPES)에 있는 기사만 [근거 N] 번호를 받고,
    나머지(검증 대상 기사)는 number=None 으로 프롬프트에는 보이지만 인용할 수 없다.
    """
    citable_types = tuple(citable_types or CITABLE_SOURCE_TYPES)   # ★ 추가
    sources, indexes = [], {}
    next_number = 1                                                # ★ 추가: 인용 가능한 기사에만 1부터 순서대로 부여
    for index, chunk in enumerate(chunks, 1):
        metadata = chunk["metadata"]
        url = normalize_url(metadata.get("url"))
        key = (f"url:{url}" if url else
               f"article:{chunk.get('article_id')}" if chunk.get("article_id") else
               f"chunk:{chunk.get('chunk_id')}" if chunk.get("chunk_id") else f"unknown:{index}")
        if key not in indexes:
            indexes[key] = len(sources)
            source_type = chunk.get("source_type") or "input"      # ★ 추가: 구분값이 없는 예전 저장본은 input 으로 간주
            number = None                                          # ★ 추가: 인용 불가 기사는 번호 없음
            if source_type in citable_types:
                number = next_number
                next_number += 1
            sources.append({
                "number": number,                                  # ★ 변경: len(sources) + 1 → 인용 가능 기사만 번호
                "source_type": source_type,                        # ★ 추가: input / related
                "title": metadata.get("title") or "미확인",
                "press": metadata.get("press") or "미확인",
                "published_at": metadata.get("published_at") or "미확인",
                "author": metadata.get("author") or "미확인",
                "url": url,
                "article_id": chunk.get("article_id"),
                "chunk_ids": [],
                "chunks": [],
            })
        source = sources[indexes[key]]
        source["chunk_ids"].append(chunk.get("chunk_id"))
        source["chunks"].append(chunk)
    return sources


def build_prompt(question, sources):
    """전체 근거를 한 번에 제공해 청크별 반복 호출과 사전 문장 필터링을 없앤다.

    ★ 변경: 검증 대상 기사(인용 불가)와 근거 기사([근거 N])를 구분해 제시한다. 발행일은 유지한다.
    """
    def block(source):
        # 번호가 없으면 검증 대상 기사, 번호가 있으면 [근거 N]
        if source["number"] is None:
            label = "[검증 대상 기사 · 인용 불가]"
        elif source["source_type"] == "input":
            label = f"[근거 {source['number']}] (검증 대상 기사)"     # CITABLE_SOURCE_TYPES 에 input 을 넣었을 때만 해당
        else:
            label = f"[근거 {source['number']}]"
        lines = [
            label,
            f"기사 제목: {source['title']}", f"언론사: {source['press']}",
            f"발행일: {source['published_at']}",
        ]
        for chunk in source["chunks"]:
            lines.append(f"청크 {chunk.get('chunk_id') or '미확인'}:\n{chunk['text'].strip()}")
        return "\n".join(lines)

    uncited = [block(s) for s in sources if s["number"] is None]
    cited = [block(s) for s in sources if s["number"] is not None]
    parts = [f"질문: {question.strip()}"]
    if uncited:
        parts.append("■ 검증 대상 기사 (주장 파악용 · 인용 금지)\n\n" + "\n\n".join(uncited))
    if cited:
        parts.append("■ 근거 기사 (인용 가능)\n\n" + "\n\n".join(cited))
    return "\n\n".join(parts)


In [ ]:
# CELL 20. 답변 자동 검사·응답용 출처 정리

def numbers(text):
    """인용 번호는 제외하고 천 단위 쉼표를 통일해 근거·답변 수치를 비교한다."""
    return {value.replace(",", "") for value in NUMBER.findall(CITATION_LIKE.sub("", text))}


def validate_answer(answer, sources):
    """도메인 규칙 없이 인용 형식·범위와 인용 근거의 숫자 포함 관계만 검사한다."""
    errors = []
    if answer.strip() == INSUFFICIENT:
        return {"auto_checks_ok": True, "errors": [], "checked_lines": 0}
    lines = [line.strip() for line in answer.strip().splitlines() if line.strip()]
    if not lines:
        return {"auto_checks_ok": False, "errors": ["빈 답변입니다."], "checked_lines": 0}
    for index, line in enumerate(lines, 1):
        block = CITATION_BLOCK.search(line)
        if not block:
            errors.append(f"{index}행: 문장 끝의 [근거 N] 인용이 없거나 형식이 잘못됐습니다.")
            continue
        cited = [int(value) for value in CITATION.findall(block.group(0))]
        invalid = sorted({value for value in cited if value < 1 or value > len(sources)})
        if invalid:
            errors.append(f"{index}행: 존재하지 않는 근거 번호 {invalid}")
            continue
        malformed = [token for token in CITATION_LIKE.findall(line) if not CITATION.fullmatch(token)]
        if malformed:
            errors.append(f"{index}행: 비표준 인용 {malformed}")
        cited_chunks = [chunk for number in cited for chunk in sources[number - 1]["chunks"]]
        source_numbers = set().union(*(numbers(chunk["text"]) for chunk in cited_chunks))
        missing = sorted(numbers(line) - source_numbers)
        if missing:
            errors.append(f"{index}행: 인용한 근거에 없는 숫자 {missing}")
    return {"auto_checks_ok": not errors, "errors": errors, "checked_lines": len(lines)}


def public_sources(sources):
    """사용자 응답에는 원문 대신 기사 메타데이터와 추적용 청크 ID만 남긴다."""
    return [{key: value for key, value in source.items() if key != "chunks"} for source in sources]


In [ ]:
# CELL 21. GPU 측정·RAGGenerator 모델 적재와 응답 생성

def gpu_memory_used_mb():
    """vLLM 작업 프로세스를 포함한 전체 GPU 사용량을 읽어 프로세스별 누락을 방지한다."""
    import shutil
    import subprocess
    if not shutil.which("nvidia-smi"):
        return None
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            check=True, capture_output=True, text=True, timeout=5,
        )
        return float(result.stdout.splitlines()[0].strip())
    except (OSError, subprocess.SubprocessError, ValueError, IndexError):
        return None

class RAGGenerator:
    """무거운 모델은 한 번만 적재하고 RAG·비교 요청을 같은 엔진으로 처리한다."""

    def __init__(self, model_name=MODEL_NAME, revision=MODEL_REVISION, backend=None):
        self.model_name = model_name
        self.revision = revision
        self.backend = backend
        self.llm = None
        self.tokenizer = None
        self.gpu_name = None

    def _load(self):
        if self.backend is not None or self.llm is not None:
            return
        import torch
        from vllm import LLM
        if not torch.cuda.is_available():
            raise RuntimeError("vLLM AWQ 실행에는 CUDA GPU가 필요합니다.")
        self.llm = LLM(
          model=self.model_name, revision=self.revision, tokenizer_revision=self.revision,
          quantization="awq", tensor_parallel_size=1, max_model_len=MAX_MODEL_LEN,
          gpu_memory_utilization=0.65,
          enforce_eager=True,
          )
        self.tokenizer = self.llm.get_tokenizer()
        self.gpu_name = torch.cuda.get_device_name(0)

    def _generate(self, messages):
        """토큰·시간·종료 이유를 같은 방식으로 기록해 cold/warm 비교를 가능하게 한다."""
        started = time.perf_counter()
        if self.backend is not None:
            result = self.backend(messages)
            text = result["text"].strip() if isinstance(result, dict) else str(result).strip()
            metrics = {"input_tokens": None, "output_tokens": None, "finish_reason": "test"}
        else:
            from vllm import SamplingParams
            self._load()
            prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            input_count = len(self.tokenizer.encode(prompt, add_special_tokens=False))
            if input_count + MAX_NEW_TOKENS > MAX_MODEL_LEN:
                raise ValueError(f"입력 {input_count} + 출력 예약 {MAX_NEW_TOKENS}가 {MAX_MODEL_LEN}을 초과합니다.")
            # nvidia-smi를 짧은 간격으로 읽어 별도 vLLM 프로세스가 쓰는 메모리도 측정한다.
            memory_before = gpu_memory_used_mb()
            memory_samples, stop_sampling = [], threading.Event()
            def sample_gpu_memory():
                while not stop_sampling.is_set():
                    value = gpu_memory_used_mb()
                    if value is not None:
                        memory_samples.append(value)
                    stop_sampling.wait(0.25)
            sampler = threading.Thread(target=sample_gpu_memory, daemon=True)
            sampler.start()
            try:
                output = self.llm.generate([prompt], SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS))[0]
            finally:
                stop_sampling.set()
                sampler.join()
            completion = output.outputs[0]
            text = completion.text.strip()
            memory_after = gpu_memory_used_mb()
            peaks = memory_samples + [value for value in (memory_before, memory_after) if value is not None]
            metrics = {
                "input_tokens": input_count,
                "output_tokens": len(getattr(completion, "token_ids", []) or []),
                "finish_reason": getattr(completion, "finish_reason", None),
                "gpu_total_memory_before_mb": memory_before,
                "gpu_total_memory_peak_mb": max(peaks) if peaks else None,
                "gpu_total_memory_after_mb": memory_after,
                "gpu_memory_scope": "whole_device_nvidia_smi_250ms_sampling",
            }
        if not text:
            raise RuntimeError("생성 모델이 빈 답변을 반환했습니다.")
        return text, {
            **metrics, "generation_time_ms": round((time.perf_counter() - started) * 1000, 2),
            "generation_call_count": 1, "model": self.model_name, "revision": self.revision,
            "gpu": self.gpu_name, "max_new_tokens": MAX_NEW_TOKENS,
            "truncated": metrics["finish_reason"] == "length",
        }

    def create_response(self, question, retrieved_chunks):
        """검색 근거가 없으면 모델을 호출하지 않고, 있으면 한 번에 종합 답변한다.
        ★ 변경: '근거'는 인용 가능한 기사(기본: 근거 기사)만 센다. 검증 대상 기사만 있으면 근거 부족으로 처리한다."""
        if not isinstance(question, str) or not question.strip():
            raise ValueError("질문은 비어 있지 않은 문자열이어야 합니다.")
        chunks = normalize_chunks(retrieved_chunks)
        sources = group_sources(chunks)
        citable = [s for s in sources if s["number"] is not None]   # ★ 추가: [근거 N] 번호가 있는 기사
        target = [s for s in sources if s["number"] is None]        # ★ 추가: 검증 대상 기사(인용 불가)
        if not citable:                                             # ★ 변경: if not chunks → if not citable
            return {
                "question": question.strip(), "answer": INSUFFICIENT, "sources": [],
                "target_articles": public_sources(target),          # ★ 추가
                "insufficient_evidence": True, "generation": None,
                "validation": {"auto_checks_ok": True, "errors": [], "checked_lines": 0},
                "fact_review": "pending_fact_review",
            }
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(question, sources)},
        ]
        answer, generation = self._generate(messages)
        validation = validate_answer(answer, citable)   # ★ 변경: 번호 1..N 이 citable 순서와 일치하도록 인용 가능 기사만 전달
        if generation["truncated"]:
            validation["errors"].append("출력 토큰 한계로 답변이 잘렸습니다.")
            validation["auto_checks_ok"] = False
        return {
            "question": question.strip(), "answer": answer, "sources": public_sources(citable),   # ★ 변경: 인용 가능 기사만
            "target_articles": public_sources(target),                                          # ★ 추가: 검증 대상 기사 정보
            "insufficient_evidence": answer == INSUFFICIENT, "generation": generation,
            "validation": validation, "fact_review": "pending_fact_review",
        }

    def create_plain_response(self, question):
        """동일 모델·디코딩의 no-RAG 기준선을 만들어 근거 제공 효과를 비교한다."""
        if not isinstance(question, str) or not question.strip():
            raise ValueError("질문은 비어 있지 않은 문자열이어야 합니다.")
        answer, generation = self._generate([
            {"role": "system", "content": PLAIN_SYSTEM_PROMPT},
            {"role": "user", "content": question.strip()},
        ])
        return {
            "question": question.strip(), "answer": answer, "sources": [],
            "generation": generation, "fact_review": "pending_fact_review",
        }


##**추상화메서드구간**
## 청킹 - 임베딩 - 구글드라이브 마운트 - 벡터검색

In [ ]:
# 청킹 임베딩 모델 생성

def create_model():
    return SentenceTransformer(
        "BAAI/bge-m3",
        model_kwargs={"torch_dtype": "float16"},
    )

In [ ]:
# CELL 22. 청킹 설정·ArticleChunker — 기사 객체 목록을 청크 목록으로 변환
#
# - class ChunkConfig
#   ➔ 청킹에 사용하는 설정값을 모아 관리하는 클래스입니다.
#   ➔ chunk_tokens: 제목 접두어를 포함한 청크의 최대 토큰 수입니다.
#   ➔ overlap_tokens: 청크 사이에 겹쳐 넣을 최대 토큰 수입니다.
#   ➔ max_tokens, reserve_tokens: 입력 한도와 특수 토큰 예산입니다.
#   ➔ split_strategy: 기본 분할 단위를 문단 또는 문장으로 지정합니다.
#   ➔ breakpoint_percentile: 의미 경계를 정할 거리의 백분위수입니다.
#   ➔ buffer_size: 의미 분석에 함께 사용할 앞뒤 문장 수입니다.
#   ➔ batch_size: 의미 분석용 임베딩의 배치 크기입니다.
#   ➔ tokenizer_name: 초기화할 모델의 이름입니다.
#
# - class ArticleChunker
#   ➔ 전처리된 ARTICLE 객체 목록을 받아 의미 기반 청크 목록을 만듭니다.
#   ➔ 외부에서는 chunk()를 호출하고, 청킹 보조 메서드는 내부에서 사용합니다.
#
# - __init__(self, model, config=None)
#   ➔ 전달받은 모델과 청킹 설정을 객체 내부에 보관합니다.
#   ➔ __model: 문맥 임베딩을 계산하는 모델입니다.
#   ➔ __tokenizer: 전달받은 모델의 토크나이저입니다.
#   ➔ __config: 청킹 설정이며, 생략하면 ChunkConfig 기본값을 사용합니다.
#
# - chunk(self, articles)
#   ➔ 외부에서 기사 목록을 청킹할 때 사용하는 공개 메서드입니다.
#   ➔ 기사마다 __chunk_article()을 호출하고 청크들을 하나의 목록으로 반환합니다.
#
# - __token_count(self, text)
#   ➔ 특수 토큰을 제외한 텍스트의 토큰 수를 계산합니다.
#
# - __split_sentences(self, text)
#   ➔ 문장 부호와 뒤따르는 공백·문자열 끝을 기준으로 문장을 나눕니다.
#   ➔ 분리한 문장에 문장 부호를 보존합니다.
#
# - __split_to_fit(self, text, limit)
#   ➔ 토큰 상한 이내의 텍스트는 그대로 반환합니다.
#   ➔ 상한을 넘으면 문장별로 나누고, 개별 문장도 넘으면 토큰 단위로 나눕니다.
#
# - __tail_within_budget(self, text, budget)
#   ➔ 다음 청크에 겹쳐 넣을 텍스트를 현재 청크의 뒤쪽에서 가져옵니다.
#   ➔ 문장을 우선 사용하고, 문장이 예산에 들어가지 않으면 뒤쪽 단어를 사용합니다.
#
# - __chunk_article(self, article)
#   ➔ 기사 한 건을 청킹하는 내부 메서드입니다.
#   ➔ 주변 문맥의 임베딩과 코사인 거리를 계산해 의미 경계를 찾습니다.
#   ➔ 의미 경계, 토큰 상한, 오버랩 설정을 적용해 청크를 구성합니다.
#   ➔ 기사 객체의 공개 속성에서 기사 ID와 메타데이터를 읽어 청크에 담습니다.

@dataclass
class ChunkConfig:
    tokenizer_name: str = "BAAI/bge-m3"  # 초기화할 임베딩 모델 이름
    max_tokens: int = 512               # 특수 토큰을 포함한 입력 한도
    chunk_tokens: int = 500             # 제목 접두어를 포함한 청크 상한
    overlap_tokens: int = 50            # 청크 사이의 최대 겹침
    reserve_tokens: int = 2             # 시작·끝 특수 토큰 자리
    split_strategy: str = "paragraph"   # 기본 단위: paragraph 또는 sentence
    breakpoint_percentile: float = 95   # 의미 경계 거리의 백분위수
    buffer_size: int = 1                # 의미 분석에 포함할 앞뒤 문장 수
    batch_size: int = 8                 # 의미 분석 임베딩 배치 크기


class ArticleChunker:
    """공개 인터페이스: chunk(articles) -> list[dict]."""

    def __init__(self, model, config=None):
        # 의존성 주입: 준비된 모델을 받아 사용하며 외부 전역 변수를 읽지 않습니다.
        self.__model = model
        self.__tokenizer = model.tokenizer
        self.__config = config if config is not None else ChunkConfig()

    def chunk(self, articles):
        # 추상화: 다른 파트는 기사 객체 목록만 전달하고 분할 과정을 알 필요가 없습니다.
        chunks = []
        for article in articles:
            chunks.extend(self.__chunk_article(article))
        return chunks

    # 은닉화: 아래 __ 메서드는 청킹 내부에서만 사용합니다.
    # 전처리의 ARTICLE 클래스는 재정의하거나 다른 파트에서 import하지 않습니다.
    def __token_count(self, text):
        return len(self.__tokenizer.encode(text, add_special_tokens=False))

    def __split_sentences(self, text):
        # 문장 부호는 보존하고, 소수점 뒤에서는 나누지 않습니다.
        return [part for part in re.split(r"(?<=[.!?。！？])(?=\s|$)", text) if part]

    def __split_to_fit(self, text, limit):
        # 상한 이내의 문장은 유지하고, 넘는 부분만 토큰 단위로 나눕니다.
        if self.__token_count(text) <= limit:
            return [text] if text else []

        parts = []
        for sentence in self.__split_sentences(text):
            ids = self.__tokenizer.encode(sentence, add_special_tokens=False)
            if len(ids) <= limit:
                parts.append(sentence)
            else:
                parts.extend(
                    self.__tokenizer.decode(
                        ids[i:i + limit], skip_special_tokens=True
                    )
                    for i in range(0, len(ids), limit)
                )
        return parts

    def __tail_within_budget(self, text, budget):
        # 문장을 우선 겹치고, 문장이 안 들어가면 뒤쪽 단어를 사용합니다.
        for parts in (
            self.__split_sentences(text),
            re.findall(r"\s*\S+", text),
        ):
            tail = ""
            for part in reversed(parts):
                if self.__token_count(part + tail) > budget:
                    break
                tail = part + tail
            if tail:
                return tail
        return ""

    def __chunk_article(self, article):
        config = self.__config
        maximum = min(config.max_tokens, self.__tokenizer.model_max_length)
        limit = min(config.chunk_tokens, maximum - config.reserve_tokens)

        prefix = f"기사제목: {article.title or ''}\n본문내용: "
        body_limit = limit - self.__token_count(prefix)

        context_limit = min(
            self.__tokenizer.model_max_length,
            self.__model.max_seq_length,
        )
        context_limit -= self.__tokenizer.num_special_tokens_to_add(pair=False)

        # 기사 객체의 공개 속성으로 기본 문단·문장과 소속 위치를 만듭니다.
        sentences, positions = [], []
        paragraphs = [
            p.strip()
            for p in re.split(r"\n\s*\n", article.body)
            if p.strip()
        ]

        for paragraph in paragraphs:
            units = (
                self.__split_sentences(paragraph)
                if config.split_strategy == "sentence"
                else [paragraph]
            )
            for unit in units:
                base = len(positions)
                for sentence in self.__split_sentences(unit):
                    parts = self.__split_to_fit(sentence, context_limit)
                    sentences.extend(parts)
                    positions.extend([base] * len(parts))

        # 주변 문장을 함께 임베딩해 인접 문맥의 코사인 거리를 계산합니다.
        boundaries = set()
        if len(sentences) > 1:
            contexts = []
            for i in range(len(sentences)):
                left = max(0, i - config.buffer_size)
                right = min(len(sentences), i + config.buffer_size + 1)

                while self.__token_count(
                    " ".join(sentences[left:right])
                ) > context_limit:
                    if i - left >= right - i - 1 and left < i:
                        left += 1
                    else:
                        right -= 1

                contexts.append(" ".join(sentences[left:right]))

            vectors = self.__model.encode(
                contexts,
                batch_size=config.batch_size,
                show_progress_bar=False,
                convert_to_numpy=True,
            ).astype("float32")

            vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
            distances = np.clip(
                1 - np.sum(vectors[:-1] * vectors[1:], axis=1),
                0,
                2,
            )
            threshold = np.percentile(
                distances, config.breakpoint_percentile
            )
            boundaries = set(
                (np.flatnonzero(distances > threshold) + 1).tolist()
            )

        # 의미 경계로 묶은 뒤 최종 청크 상한을 적용합니다.
        units, breaks = [], set()
        group, previous_base = "", None

        for i, sentence in enumerate(sentences):
            if group and (positions[i] != previous_base or i in boundaries):
                units.extend(self.__split_to_fit(group, body_limit))
                group = ""

            if i in boundaries:
                breaks.add(len(units))

            group += sentence
            previous_base = positions[i]

        units.extend(self.__split_to_fit(group, body_limit))

        # 의미 경계 또는 토큰 상한에서 청크를 확정하고 오버랩을 붙입니다.
        segments, current = [], ""
        for i, unit in enumerate(units):
            glue = (
                " "
                if current
                and not current[-1].isspace()
                and not unit[0].isspace()
                else ""
            )

            if (
                i not in breaks
                and self.__token_count(current + glue + unit) <= body_limit
            ):
                current += glue + unit
                continue

            if current:
                segments.append(current)

            tail = self.__tail_within_budget(
                current, config.overlap_tokens
            )
            glue = (
                " "
                if tail
                and not tail[-1].isspace()
                and not unit[0].isspace()
                else ""
            )
            current = (
                tail + glue + unit
                if self.__token_count(tail + glue + unit) <= body_limit
                else unit
            )

        if current:
            segments.append(current)

        # 기사 ID를 새로 계산하지 않고 전처리 객체가 제공하는 값을 사용합니다.
        article_id = article.url

        return [{
            "chunk_id": f"{article_id}_chunk{i:04d}",
            "article_id": article_id,
            "chunk_index": i,
            "text": text,
            "token_count": self.__token_count(text),
            "embedding_token_count": self.__token_count(prefix + text),
            "metadata": {
                "title": article.title,
                "url": article.url,
                "press": article.press,
                "published_at": article.published_at,
                "warnings": list(article.warnings),
            },
        } for i, text in enumerate(segments)]

In [ ]:
# CELL 23. ChunkEmbedder — 청크를 임베딩·벡터 스토어로 변환
#
# *** 의존성 ***
# (CELL 11. ArticleChunker: get_model())
# model 중복 생성 방지
#
# - class ChunkEmbedder
#   ➔ 청크 목록으로부터 임베딩 배열과 FAISS 벡터 스토어를 만듭니다.
#   ➔ 외부에서는 embed()를 호출해 결과를 받습니다.
#
# - __init__(self, model, model_name="BAAI/bge-m3", batch_size=16)
#   ➔ 사용할 모델, 저장할 모델 이름, 임베딩 배치 크기를 보관합니다.
#   ➔ __model: 청크를 임베딩하는 모델입니다.
#   ➔ __model_name: 벡터 스토어에 기록할 모델 이름입니다.
#   ➔ __batch_size: 한 번에 임베딩할 청크 수입니다.
#
# - embed(self, chunks)
#   ➔ 청크 본문에 기사 제목 접두어를 붙여 임베딩 입력을 만듭니다.
#   ➔ 모델 출력을 float32로 변환하고 L2 정규화를 적용합니다.
#   ➔ 내적 기반 SQ8 인덱스를 생성하고 학습한 뒤 벡터를 추가합니다.
#   ➔ 임베딩 배열과 vector_store를 함께 반환합니다.
#   ➔ 임베딩 행, 인덱스 위치, 청크 목록의 순서를 동일하게 유지합니다.
#   ➔ vector_store에는 index, chunks, embedding_model, chunk_ids가 들어갑니다.

import faiss
import numpy as np

class ChunkEmbedder:
    """공개 인터페이스: embed(chunks) -> (embeddings, vector_store)."""

    def __init__(self, model, model_name="BAAI/bge-m3", batch_size=16):
        # 모델과 설정을 객체 내부에 보관하고 다른 파트의 전역 변수를 읽지 않습니다.
        self.__model = model
        self.__model_name = model_name
        self.__batch_size = batch_size

    def embed(self, chunks):
        # 추상화: 호출자는 정규화·SQ8 생성 과정을 직접 실행하지 않습니다.
        # 제목 접두어 형식은 청킹에서 토큰 수를 계산한 형식과 같습니다.
        texts = [
            f"기사제목: {c['metadata'].get('title') or ''}\n본문내용: {c['text']}"
            for c in chunks
        ]

        dimension = self.__model.get_sentence_embedding_dimension()
        embeddings = np.empty((0, dimension), dtype="float32")

        index = faiss.IndexFlatIP(dimension)

        if texts:
            # FP16 모델 출력 → float32 변환 → L2 정규화 → SQ8 학습·추가
            embeddings = self.__model.encode(
                texts,
                batch_size=self.__batch_size,
                show_progress_bar=True,
                convert_to_numpy=True,
            ).astype("float32")

            faiss.normalize_L2(embeddings)
            index.train(embeddings)
            index.add(embeddings)

        # 출력 계약: embeddings의 i번째 행과 chunks의 i번째 항목이 대응합니다.
        vector_store = {
            "index": index,
            "chunks": list(chunks),
            "embedding_model": self.__model_name,
            "chunk_ids": [c["chunk_id"] for c in chunks],
        }
        return embeddings, vector_store

In [ ]:
# CELL 24. VectorStoreManager — Google Drive 누적 저장
#
# *** 의존성 ***
# (CELL 14. VectorLoadManager)
#
# - class VectorStoreManager
#   ➔ 지정한 Google Drive 경로에 벡터 스토어를 누적 저장합니다.
#   ➔ 외부에서는 save()를 호출하고, 파일 읽기·쓰기 과정은 내부에서 처리합니다.
#
# - __init__(self, save_dir)
#   ➔ 저장 폴더와 인덱스·청크·설정 파일의 경로를 보관합니다.
#   ➔ __directory: 저장 폴더의 경로입니다.
#   ➔ __index_path: article_vector.index의 경로입니다.
#   ➔ __chunks_path: article_chunks.json의 경로입니다.
#   ➔ __config_path: vector_config.json의 경로입니다.
#
# - save(self, vector_store, embeddings)
#   ➔ 외부에서 이번 처리 결과를 저장할 때 사용하는 공개 메서드입니다.
#   ➔ 입력 청크를 복사하고 청크 수와 임베딩 배열의 크기를 확인합니다.
#   ➔ 기존 인덱스가 있으면 VectorLoadManager.load()로 저장본을 불러옵니다.
#   ➔ 기존 인덱스가 없으면 전달받은 인덱스를 복제하고 비웁니다.
#   ➔ 다른 기사의 데이터는 유지하며 청크와 인덱스의 순서를 맞춥니다.
#   ➔ 누적 인덱스, 청크 목록, 모델·차원·청크 순서 정보를 파일에 기록합니다.
#   ➔ 이전 기사까지 포함한 누적 vector_store를 반환합니다.

class VectorStoreManager:
    """공개 인터페이스: save(vector_store, embeddings) -> 누적 vector_store."""

    def __init__(self, save_dir):
        # 은닉화: 파일 이름과 경로 구성은 클래스 내부에서 관리합니다.
        self.__directory = Path(save_dir)
        self.__index_path = self.__directory / "article_vector.index"
        self.__chunks_path = self.__directory / "article_chunks.json"
        self.__config_path = self.__directory / "vector_config.json"

    def save(self, vector_store, embeddings):
        # 입력을 복사해 호출자의 청크와 저장 결과를 분리합니다.
        new_chunks = deepcopy(vector_store["chunks"])
        new_vectors = np.ascontiguousarray(embeddings, dtype="float32")

        if new_vectors.shape != (len(new_chunks), vector_store["index"].d):
            raise ValueError("청크 수·벡터 수·차원이 일치해야 합니다.")

        if self.__index_path.is_file():
            # 같은 파트의 공개 메서드로 기존 저장본을 불러옵니다.
            saved = VectorLoadManager(self.__directory).load()
            index, chunks = saved["index"], saved["chunks"]

            # 이미 저장된 기사 ID에 속한 청크는 모두 건너뜁니다.
            known_article_ids = {c["article_id"] for c in chunks}
            keep = [
                i for i, c in enumerate(new_chunks)
                if c["article_id"] not in known_article_ids
            ]

            # 신규 기사가 없으면 파일을 다시 쓰지 않습니다.
            if not keep:
                return saved

            if (
                saved["embedding_model"] != vector_store["embedding_model"]
                or index.d != new_vectors.shape[1]
            ):
                raise ValueError("저장본과 임베딩 모델·차원이 다릅니다.")

            # 같은 위치 번호로 청크와 벡터를 골라 대응 순서를 유지합니다.
            new_chunks = [new_chunks[i] for i in keep]
            new_vectors = new_vectors[keep]

        else:
            # 최초 저장에서는 전달받은 인덱스 구조를 복제하고 비웁니다.
            index = faiss.clone_index(vector_store["index"])
            index.reset()
            chunks = []

        # 기존 데이터는 유지하고 신규 기사의 벡터·청크만 추가합니다.
        if new_chunks:
            if not index.is_trained:
                index.train(new_vectors)
            index.add(new_vectors)

        chunks.extend(new_chunks)

        config = {
            "embedding_model": vector_store["embedding_model"],
            "dimension": index.d,
            "vector_count": index.ntotal,
            "chunk_ids": [c["chunk_id"] for c in chunks],
        }

        # 추상화: 인덱스·청크·설정의 파일 저장을 내부에서 처리합니다.
        faiss.write_index(index, str(self.__index_path))

        for path, data in (
            (self.__chunks_path, chunks),
            (self.__config_path, config),
        ):
            with path.open("w", encoding="utf-8") as file:
                json.dump(data, file, ensure_ascii=False, indent=2)

        return {
            "index": index,
            "chunks": chunks,
            "embedding_model": config["embedding_model"],
            "chunk_ids": config["chunk_ids"],
        }

In [ ]:
# CELL 25. VectorLoadManager (임시) — Google Drive 저장본 불러오기
#
# - class VectorLoadManager
#   ➔ 지정한 Google Drive 경로에서 저장된 벡터 스토어를 불러옵니다.
#   ➔ 외부에서는 load()를 호출하고, 파일 형식별 복원은 내부에서 처리합니다.
#
# - __init__(self, save_dir)
#   ➔ 불러올 폴더와 인덱스·청크·설정 파일의 경로를 보관합니다.
#   ➔ __directory: 저장본이 위치한 폴더의 경로입니다.
#   ➔ __index_path: 읽을 FAISS 인덱스 파일의 경로입니다.
#   ➔ __chunks_path: 읽을 청크 JSON 파일의 경로입니다.
#   ➔ __config_path: 읽을 설정 JSON 파일의 경로입니다.
#
# - load(self)
#   ➔ FAISS 인덱스와 청크·설정 JSON을 읽습니다.
#   ➔ 벡터 수, 청크 수, 벡터 차원, 청크 ID 순서가 일치하는지 확인합니다.
#   ➔ 서로 맞지 않으면 오류를 발생시켜 불일치한 저장본의 사용을 막습니다.
#   ➔ index, chunks, embedding_model, chunk_ids를 가진 vector_store를 반환합니다.
#   ➔ 모델 로딩이나 임베딩 재계산 없이 저장된 결과를 복원합니다.


class VectorLoadManager:
    """공개 인터페이스: load() -> vector_store."""

    def __init__(self, save_dir):
        self.__directory = Path(save_dir)
        self.__index_path = self.__directory / "article_vector.index"
        self.__chunks_path = self.__directory / "article_chunks.json"
        self.__config_path = self.__directory / "vector_config.json"

    def load(self):
        # 추상화: 호출자는 파일 형식이나 FAISS 복원 방법을 알 필요가 없습니다.
        index = faiss.read_index(str(self.__index_path))

        with self.__chunks_path.open("r", encoding="utf-8") as file:
            chunks = json.load(file)

        with self.__config_path.open("r", encoding="utf-8") as file:
            config = json.load(file)

        # 다른 파트에 서로 맞지 않는 인덱스·청크가 전달되는 것을 막습니다.
        chunk_ids = [c["chunk_id"] for c in chunks]
        if (
            index.ntotal != len(chunks)
            or index.ntotal != config["vector_count"]
            or index.d != config["dimension"]
            or chunk_ids != config["chunk_ids"]
        ):
            raise ValueError(
                "저장된 인덱스·청크·구성 정보가 일치하지 않습니다."
            )

        # 임베딩·저장 클래스와 동일한 공개 출력 형식으로 반환합니다.
        return {
            "index": index,
            "chunks": chunks,
            "embedding_model": config["embedding_model"],
            "chunk_ids": chunk_ids,
        }

In [ ]:
# CELL 26. VectorSearchManager — 질문과 의미가 유사한 기사 청크 검색
#
# - class VectorSearchManager
#   ➔ 질문 임베딩·L2 정규화·FAISS 검색·기사 필터링을 내부에서 처리합니다.
#   ➔ 외부에서는 search()를 호출해 검색 결과를 받습니다.
#
# - __init__(self, model)
#   ➔ 문서를 임베딩할 때 사용한 모델을 전달합니다.
#   ➔ __model: 질문을 벡터로 변환하는 모델입니다.
#
# - search(self, query, vector_store, article_ids=None, top_k=4, min_score=0.2)
#   ➔ query: 검색할 질문입니다.
#   ➔ vector_store: 검색 시점의 인덱스와 청크 목록입니다.
#   ➔ article_ids: 검색을 허용할 기사 ID 목록입니다.
#      None이면 전체 기사에서 검색하고, 빈 목록이면 빈 결과를 반환합니다.
#   ➔ top_k: 반환할 최대 청크 수입니다.
#   ➔ min_score: 검색 결과에 포함할 최소 내적 점수입니다.
#   ➔ 원본 청크와 메타데이터에 점수·구분을 추가한 목록을 반환합니다.
#   ➔ 기존 UI 검색 함수의 반환 형식을 유지합니다.

class VectorSearchManager:
    """공개 인터페이스: search() -> 검색 결과 목록."""

    def __init__(self, model):
        # 모델을 내부에 보관하고 다른 셀의 전역 변수를 직접 읽지 않습니다.
        self.__model = model

    def search(
        self,
        query,
        vector_store,
        article_ids=None,
        top_k=4,
        min_score=0.2,
    ):
        if not isinstance(query, str) or not query.strip():
            raise ValueError("query는 비어 있지 않은 문자열이어야 합니다.")

        if not isinstance(top_k, int) or isinstance(top_k, bool) or top_k < 1:
            raise ValueError("top_k는 1 이상의 정수여야 합니다.")

        # None은 전체 검색, 빈 목록은 검색할 기사가 없다는 의미입니다.
        allowed = None if article_ids is None else set(article_ids)
        if vector_store is None or allowed == set():
            return []

        # 최신 누적 저장소를 호출할 때마다 전달받아 사용합니다.
        index = vector_store["index"]
        chunks = vector_store["chunks"]

        if index is None or index.ntotal == 0:
            return []

        if index.ntotal != len(chunks):
            raise ValueError("인덱스의 벡터 수와 청크 수가 일치하지 않습니다.")

        # 질문을 문서와 동일한 모델로 임베딩하고 float32로 변환합니다.
        query_embedding = self.__model.encode(
            [query],
            convert_to_numpy=True,
            show_progress_bar=False,
        ).astype("float32")

        # 저장된 문서 벡터와 동일하게 L2 정규화를 적용합니다.
        # 내적 기반 SQ8 인덱스에서는 코사인 유사도의 근삿값을 얻습니다.
        faiss.normalize_L2(query_embedding)

        # 기사 필터가 있으면 전체 후보를 검색한 뒤 기사 ID로 거릅니다.
        # 상위 후보가 다른 기사라는 이유로 대상 기사 청크가 누락되는 것을 막습니다.
        candidate_k = (
            index.ntotal
            if allowed is not None
            else min(top_k, index.ntotal)
        )
        scores, indices = index.search(query_embedding, candidate_k)

        results = []
        for score, chunk_idx in zip(scores[0], indices[0]):
            if chunk_idx < 0:
                continue

            # 내적 점수는 내림차순이므로 최소 점수 미만이면 중단합니다.
            if float(score) < min_score:
                break

            # 인덱스 위치와 같은 순서로 저장된 원문 청크를 가져옵니다.
            chunk = chunks[int(chunk_idx)]

            if allowed is not None and chunk.get("article_id") not in allowed:
                continue

            # 기존 UI와 RAG가 사용하는 반환 형식을 유지합니다.
            # 새 딕셔너리를 만들어 저장소의 원본 청크는 수정하지 않습니다.
            metadata = chunk.get("metadata") or {}
            results.append({
                **metadata,
                **chunk,
                "score": float(score),
                "retrieval_group": "related",
                "source_type": "related",
            })

            if len(results) >= top_k:
                break

        return results

##**RAG LOAD**

In [ ]:
# 생성기 객체를 만든 뒤, 모델을 미리 GPU에 적재(사전 로딩)한다.
# - RAGGenerator(): 기본값(MODEL_NAME, MODEL_REVISION)으로 객체만 생성. 이 시점에는 모델이 아직 로드되지 않음(지연 로딩).
# - generator._load(): vLLM 엔진과 토크나이저를 지금 적재.
#   → 첫 질문 때 모델 적재 시간이 섞이지 않게 해서 cold/warm 시간 측정을 공정하게 만든다.
# 주의: CUDA GPU가 없으면 RuntimeError가 발생하고, 적재에는 수 분이 걸릴 수 있다.
generator = RAGGenerator(); generator._load()

##**UI 사용 시 들어갈 파이프라인**
## 추상화 메서드 기능 사용구간

In [ ]:
# ═══════════════ 근거 수집 파이프라인 (전처리 → 청킹 → 임베딩 → 벡터 DB) ═══════════════
# 실행 순서: CELL 11(ArticleChunker) → 12(ChunkEmbedder) → 13(VectorStoreManager)
#            → 14(VectorLoadManager) → 이 셀 → UI 셀
# 이 셀은 청킹·임베딩·저장 클래스를 "재정의하지 않고" 위 셀의 클래스를 그대로 조립합니다.
ProgressFn = Callable[[int, Optional[str]], None]


def _noop(percent: int, stage: Optional[str] = None) -> None:
    pass


# ───────────── 0) 수집·전처리 ─────────────
@dataclass
class CollectResult:
    """어떤 수집원을 쓰든 파이프라인이 받는 공통 결과 형식."""
    question: dict                                          # {"keywords": [...], "queries": [...]}
    articles: list = field(default_factory=list)            # 이번에 새로 수집한 기사
    reused_article_ids: set = field(default_factory=set)    # 이미 DB에 있어 연결만 할 기사


class KeywordExtractor:
    """질문 → 검색 키워드. 분석기·개수 정책을 한곳에서 관리."""

    def __init__(self, analyzer, max_keywords, multiplier=3):
        self.analyzer = analyzer
        self.top_k = max_keywords * multiplier

    def __call__(self, text):
        return extract_question_keywords(text, self.analyzer, top_k=self.top_k)

def article_key(art):
    """Article에는 article_id가 없으므로 정규화된 URL을 ID로 사용"""
    return getattr(art, "article_id", None) or art.url
class ArticleSource(ABC):
    """기사 수집원 인터페이스. 다른 수집원을 붙일 때 collect()만 구현하면 된다."""

    @abstractmethod
    def collect(self, text, keywords, known_article_ids) -> CollectResult: ...


class NaverNewsSource(ArticleSource):
    """CELL 9의 collect_articles()를 그대로 호출하는 어댑터."""

    def collect(self, text, keywords, known_article_ids):
        raw = collect_articles(text, known_urls=known_article_ids)
        return CollectResult(
            question=raw["question"],
            articles=raw["related_articles"],
            reused_article_ids=set(raw["reused_urls"]),
        )



# 추가 정제 단계: article -> article(수정본) 또는 None(버림)
ArticleCleaner = Callable[[object], Optional[object]]

def apply_cleaners(articles, cleaners):
    out = []
    for art in articles:
        for clean in cleaners:
            art = clean(art)
            if art is None:
                break
        if art is not None:
            out.append(art)
    return out


# ───────────── 1) 벡터 저장소 (CELL 13·14 래퍼) ─────────────
class VectorRepository:
    """누적 vector_store를 메모리에 들고 있고, 저장은 VectorStoreManager에 맡긴다.

    - 시작할 때 Drive에 저장본이 있으면 VectorLoadManager로 복원한다.
    - add()는 이번에 새로 만든 (vector_store, embeddings)를 save()에 넘기고,
      반환된 '누적 vector_store'로 내부 상태를 교체한다.
    """

    def __init__(self, save_dir):
        self._manager = VectorStoreManager(save_dir)
        self.store = None
        if (Path(save_dir) / "article_vector.index").is_file():
            self.store = VectorLoadManager(save_dir).load()

    def known_article_ids(self):
        return {c["article_id"] for c in self.store["chunks"]} if self.store else set()

    def add(self, new_store, embeddings):
        self.store = self._manager.save(new_store, embeddings)


# ───────────── 2) 전체 파이프라인 ─────────────
@dataclass
class IngestResult:
    used: list = field(default_factory=list)       # 청킹에 성공한 기사
    failed: list = field(default_factory=list)     # "url: 사유"
    new_chunks: list = field(default_factory=list)



class EvidencePipeline:
    """질문 → 키워드 → 기사 수집 → 추가 정제 → 청킹(CELL 11) → 임베딩(CELL 12) → 저장(CELL 13).

    각 단계는 생성자로 주입한다. UI는 run() 한 줄만 호출하면 된다.
    """

    def __init__(self, keyword_extractor, source, chunker, embedder, repository, cleaners=()):
        self.keyword_extractor = keyword_extractor  # (text) -> list[str]
        self.source = source                        # ArticleSource
        self.cleaners = list(cleaners)
        self.chunker = chunker                      # ArticleChunker (chunk(list) 사용)
        self.embedder = embedder                    # ChunkEmbedder (embed(chunks) 사용)
        self.repository = repository                # VectorRepository

    # 청킹→임베딩→저장만 따로 쓰고 싶을 때(예: 업로드 문서 색인)도 호출 가능
    def ingest(self, articles, on_progress: ProgressFn = _noop) -> IngestResult:
        res = IngestResult()

        # 같은 실행 안에서 article_id가 겹치는 기사와 이미 저장된 기사는 제외
        # (VectorStoreManager.save는 저장본과의 중복만 거르므로 배치 내부 중복은 여기서 처리)
        seen = self.repository.known_article_ids()
        unique = []
        for art in articles:
            if article_key(art) in seen: continue
            seen.add(article_key(art))
            unique.append(art)

        on_progress(40, f"본문 청킹 중 (기사 {len(unique)}건)")
        for n, art in enumerate(unique, 1):
            try:                                    # 기사 하나가 실패해도 나머지는 진행
                chunks = self.chunker.chunk([art])
                if not chunks:
                    raise ValueError("청크가 생성되지 않았습니다. (본문이 비었거나 문장 분리 결과가 없음)")
                res.new_chunks.extend(chunks)
                res.used.append(art)
            except Exception as exc:
                res.failed.append(f"{getattr(art, 'url', '?')}: {exc}")
            on_progress(40 + int(10 * n / len(unique)), None)

        if not res.new_chunks:
            return res

        on_progress(50, f"임베딩 중 (청크 {len(res.new_chunks)}개)")
        embeddings, new_store = self.embedder.embed(res.new_chunks)

        on_progress(76, "벡터 DB에 저장 중")
        self.repository.add(new_store, embeddings)
        return res

    def run(self, text, qid, on_progress: ProgressFn = _noop) -> dict:
        #{article_key(a) for a in fresh}
        on_progress(5, "입력 문장에서 키워드 추출 중")
        keywords = self.keyword_extractor(text)
        if not keywords:
            raise ValueError("입력 문장에서 검색에 쓸 키워드를 뽑지 못했습니다. 조금 더 구체적으로 입력해 주세요.")

        on_progress(12, "뉴스 검색·관련 기사 수집 중")
        try:
            got = self.source.collect(text, keywords, self.repository.known_article_ids())
        except Exception as exc:
            traceback.print_exc()
            raise RuntimeError(f"기사 수집 실패: {exc}") from exc

        fresh = apply_cleaners(got.articles, self.cleaners)
        res = self.ingest(fresh, on_progress)

        # 질문-기사 연결: 청크를 수정하지 않고 article_id 목록으로 돌려준다.
        # (검색 시 이 article_id로 필터링하면 '이 질문의 근거'만 볼 수 있다)
                # 수집된 기사 중 실제로 DB에 들어 있는 것만 이 질문의 근거로 연결한다.
        # (이미 저장돼 ingest에서 건너뛴 기사도 포함, 청킹 실패 기사는 제외)
        stored = self.repository.known_article_ids()
        article_ids = sorted(
            (set(got.reused_article_ids) | {a.url for a in fresh}) & stored)

        return {
            "qid": qid,
            "keywords": got.question["keywords"], "queries": got.question["queries"],
            "found": len(fresh) + len(got.reused_article_ids), "new_articles": len(res.used),
            "failed": res.failed,
            "article_ids": article_ids,                 # 이 질문에 연결된 기사 ID
            "_used_articles": res.used,                 # 전역 상태 반영용(호출부에서 꺼내 쓰고 지움)
        }


# ───────────── 조립 예시 ─────────────
# chunker  = ArticleChunker(model, ChunkConfig(chunk_tokens=CHUNK_TOKENS, overlap_tokens=OVERLAP_TOKENS,
#                                              split_strategy=SPLIT_STRATEGY))
# embedder = ChunkEmbedder(model, model_name="BAAI/bge-m3", batch_size=16)
# repo     = VectorRepository(SAVE_DIR)
# pipeline = EvidencePipeline(
#     keyword_extractor=KeywordExtractor(analyzer, MAX_KEYWORDS),
#     source=NaverNewsSource(naver_config),
#     chunker=chunker, embedder=embedder, repository=repo)
# result = pipeline.run(text, qid, on_progress=...)
# vector_store = repo.store        # 검색에는 항상 repo.store를 사용

## 챗봇 서비스 연동

In [ ]:
# ═══════════════ 답변 평가 (일반 LLM vs 근거 기반 RAG) ═══════════════
# cell_pipeline.py 다음, UI 셀보다 "앞"에서 실행하세요.
# 핵심 아이디어: 두 답변을 "같은 근거 구절"에 대고 채점해서 같은 잣대로 비교한다.

EVAL_CONFIG = {
    # 답변 문장이 근거 구절과 이 값 이상 유사하면 "근거가 뒷받침한다"고 본다.
    # 임베딩 모델마다 유사도 분포가 다르므로 처음 몇 건을 돌려 보고 조정하세요.
    "support_threshold": 0.55,
    "min_claim_chars": 8,                       # 이보다 짧은 문장은 채점에서 제외
    # RAG 종합점수 가중치(편의용 요약값입니다. 개별 지표를 함께 보세요)
    "weights": {"support_rate": 0.4, "citation_precision": 0.3,
                "citation_coverage": 0.2, "answer_relevancy": 0.1},
    "log_path": "eval_results.jsonl",           # 질문마다 한 줄씩 누적 저장
}

# [근거 1] / [근거 1, 2] / [근거 1][근거 2] 형식을 모두 인식. CELL 15의 인용 형식과 다르면 여기를 수정.
_CITE_RE = re.compile(r"\[\s*근거\s*([\d\s,，]+?)\s*\]")
_SENT_SPLIT = re.compile(r"(?<=[.!?。])\s+")
_BULLET_RE = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s*")


# ───────────── 기본 도구 ─────────────
def parse_claims(answer, min_chars=8):
    """답변 → [{"text": 문장, "cites": [근거 번호]}]. 같은 줄의 문장은 그 줄의 인용을 공유한다."""
    claims, short = [], []
    for line in (answer or "").splitlines():
        line = line.strip()
        if not line:
            continue
        cites = list(dict.fromkeys(int(n) for m in _CITE_RE.findall(line) for n in re.findall(r"\d+", m)))
        clean = _BULLET_RE.sub("", _CITE_RE.sub("", line)).strip()
        for sent in _SENT_SPLIT.split(clean):
            sent = sent.strip()
            if not sent:
                continue
            (claims if len(sent) >= min_chars else short).append({"text": sent, "cites": cites})
    # "네, 있습니다."처럼 짧은 문장만 있는 답변은 버리지 않고 그대로 채점한다
    return claims or short


def _embed(model, texts):
    v = model.encode(texts, batch_size=16, convert_to_numpy=True, show_progress_bar=False).astype("float32")
    norm = np.linalg.norm(v, axis=1, keepdims=True)
    norm[norm == 0] = 1.0
    return v / norm


def _bigrams(s):
    s = re.sub(r"\s+", "", s or "")
    return [s[i:i + 2] for i in range(len(s) - 1)]


def char_f1(pred, ref):
    """글자 2-gram F1 (정답 문장이 있을 때만 사용). 형태소 분석기 없이 한국어에 무난하게 동작."""
    a, b = Counter(_bigrams(pred)), Counter(_bigrams(ref))
    overlap = sum((a & b).values())
    if not overlap:
        return 0.0
    p, r = overlap / sum(a.values()), overlap / sum(b.values())
    return 2 * p * r / (p + r)


def _build_evidence(retrieved, model):
    texts = [(c.get("text") or "").strip() for c in retrieved]
    keep = [i for i, t in enumerate(texts) if t]
    if not keep:
        return {"emb": None, "texts": [], "rows": {}}
    return {"emb": _embed(model, [texts[i] for i in keep]),
            "texts": [texts[i] for i in keep],
            "rows": {retrieved[i].get("chunk_id"): r for r, i in enumerate(keep)}}


# ───────────── 지표 계산 ─────────────
def _score_claims(claims, ev, model, cfg, judge=None):
    """문장별로 가장 비슷한 근거 구절을 찾아 '뒷받침 여부'를 판정. (지표 dict, 유사도 행렬) 반환."""
    out = {"n_claims": len(claims), "support_rate": None, "mean_support": None, "unsupported_claims": []}
    if not claims or ev["emb"] is None:
        return out, None
    sims = _embed(model, [c["text"] for c in claims]) @ ev["emb"].T
    best, best_idx = sims.max(axis=1), sims.argmax(axis=1)
    ok = []
    for i, c in enumerate(claims):
        if judge:                                   # 선택: LLM/NLI 판정 함수 (claim, evidence) -> bool
            ok.append(bool(judge(c["text"], ev["texts"][best_idx[i]])))
        else:
            ok.append(bool(best[i] >= cfg["support_threshold"]))
    out["support_rate"] = sum(ok) / len(claims)
    out["mean_support"] = float(best.mean())
    out["unsupported_claims"] = [claims[i]["text"][:120] for i, v in enumerate(ok) if not v][:5]
    return out, sims


def _citation_metrics(claims, sims, rag, ev, thr):
    """RAG 전용. 인용 포함 비율 / 인용 번호 유효율 / 인용 정확도(인용한 출처가 실제로 그 문장을 뒷받침하는가)."""
    sources = {s.get("number"): [ev["rows"][cid] for cid in s.get("chunk_ids", []) if cid in ev["rows"]]
               for s in rag.get("sources", [])}
    out = {"citation_coverage": None, "citation_valid_rate": None, "citation_precision": None}
    if not claims:
        return out
    cited = [(i, c) for i, c in enumerate(claims) if c["cites"]]
    out["citation_coverage"] = len(cited) / len(claims)
    nums = {n for _, c in cited for n in c["cites"]}
    if nums:
        out["citation_valid_rate"] = sum(n in sources for n in nums) / len(nums)
    if cited and sims is not None:
        good = 0
        for i, c in cited:
            rows = sorted({r for n in c["cites"] for r in sources.get(n, [])})
            if rows and sims[i, rows].max() >= thr:
                good += 1
        out["citation_precision"] = good / len(cited)
    return out


def _relevancy(model, question, claims):
    if not claims:
        return None
    q, a = _embed(model, [question, " ".join(c["text"] for c in claims)])
    return float(q @ a)


def _reference_metrics(model, answer, reference):
    if not reference or not (answer or "").strip():
        return {"ref_f1": None, "ref_sim": None}
    a, r = _embed(model, [answer, reference])
    return {"ref_f1": round(char_f1(answer, reference), 4), "ref_sim": round(float(a @ r), 4)}


def _overall(metrics, weights):
    """값이 있는 지표만으로 가중 평균(가중치는 재정규화)."""
    pairs = [(weights[k], metrics[k]) for k in weights if metrics.get(k) is not None]
    if not pairs:
        return None
    return sum(w * v for w, v in pairs) / sum(w for w, _ in pairs)


def _round(d):
    return {k: (round(v, 4) if isinstance(v, float) else v) for k, v in d.items()}


# ───────────── 한 질문 평가 ─────────────
def evaluate_case(question, rag, retrieved, plain, model, reference=None, cfg=None, judge=None):
    """
    question  : 질문 문자열
    rag       : RAGGenerator.create_response 결과 (answer, sources, validation, insufficient_evidence ...)
    retrieved : 검색된 청크 목록 (text, chunk_id, score ...)
    plain     : create_plain_response 결과 또는 None
    reference : (선택) 정답 문장. 있으면 ref_f1 / ref_sim 도 계산
    """
    cfg = {**EVAL_CONFIG, **(cfg or {})}
    ev = _build_evidence(retrieved, model)
    thr = cfg["support_threshold"]
    insufficient = bool(rag.get("insufficient_evidence"))
    gen = rag.get("generation") or {}
    val = rag.get("validation") or {}

    # ── RAG 답변
    rag_claims = [] if insufficient else parse_claims(rag.get("answer", ""), cfg["min_claim_chars"])
    rag_m, sims = _score_claims(rag_claims, ev, model, cfg, judge)
    rag_m.update(_citation_metrics(rag_claims, sims, rag, ev, thr))
    rag_m["answer_relevancy"] = _relevancy(model, question, rag_claims)
    scores = [c.get("score") for c in retrieved if isinstance(c.get("score"), (int, float))]
    rag_m.update({
        "insufficient": insufficient,
        "validator_ok": bool(val.get("auto_checks_ok", True)),
        "validator_errors": len(val.get("errors", [])),
        "n_sources": len(rag.get("sources", [])),
        "n_chunks": len(retrieved),
        "retrieval_mean_score": float(np.mean(scores)) if scores else None,
        "generation_ms": gen.get("generation_time_ms"),
    })
    rag_m.update(_reference_metrics(model, rag.get("answer", ""), reference))
    rag_m["overall"] = None if insufficient else _overall(rag_m, cfg["weights"])

    # ── 일반 LLM 답변 (같은 근거에 대고 채점)
    plain_m = None
    if plain:
        p_claims = parse_claims(plain.get("answer", ""), cfg["min_claim_chars"])
        plain_m, _ = _score_claims(p_claims, ev, model, cfg, judge)
        plain_m["answer_relevancy"] = _relevancy(model, question, p_claims)
        plain_m["generation_ms"] = (plain.get("generation") or {}).get("generation_time_ms")
        plain_m.update(_reference_metrics(model, plain.get("answer", ""), reference))

    # ── 비교
    cmp = {"support_gain": None, "plain_unsupported_rate": None}
    if plain_m and rag_m["support_rate"] is not None and plain_m["support_rate"] is not None:
        cmp["support_gain"] = rag_m["support_rate"] - plain_m["support_rate"]
    if plain_m and plain_m["support_rate"] is not None:
        cmp["plain_unsupported_rate"] = 1 - plain_m["support_rate"]   # 근거로 확인되지 않는 문장 비율(환각 위험 대용치)

    return {"question": question, "reference": reference,
            "rag": _round(rag_m), "plain": _round(plain_m) if plain_m else None,
            "comparison": _round(cmp), "support_threshold": thr}


# ───────────── 누적 기록·요약 ─────────────
_METRICS = ["support_rate", "mean_support", "answer_relevancy", "citation_precision",
            "citation_coverage", "citation_valid_rate", "generation_ms", "ref_f1", "ref_sim"]


def flatten(rec):
    """기록 1건을 CSV 한 줄(평평한 dict)로."""
    row = {"question": rec["question"], "reference": rec.get("reference") or ""}
    for side in ("rag", "plain"):
        for k, v in (rec.get(side) or {}).items():
            if not isinstance(v, (list, dict)):
                row[f"{side}.{k}"] = v
    for k, v in (rec.get("comparison") or {}).items():
        row[f"cmp.{k}"] = v
    return row


class EvalLog:
    def __init__(self, path=None):
        self.records, self.path, self._lock = [], path, threading.Lock()

    def add(self, rec):
        with self._lock:
            self.records.append(rec)
        if self.path:
            try:
                with open(self.path, "a", encoding="utf-8") as f:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            except Exception:
                pass                                   # 저장 실패가 서비스를 막지 않도록

    @staticmethod
    def _mean(vals):
        vals = [float(v) for v in vals if v is not None and not isinstance(v, bool)]
        return round(sum(vals) / len(vals), 4) if vals else None

    def summary(self):
        with self._lock:
            recs = list(self.records)
        rag = [r["rag"] for r in recs]
        plain = [r["plain"] for r in recs if r.get("plain")]
        both = [(r["rag"]["support_rate"], r["plain"]["support_rate"]) for r in recs
                if r.get("plain") and r["rag"]["support_rate"] is not None and r["plain"]["support_rate"] is not None]
        return {
            "n": len(recs),
            "rag": {**{k: self._mean(x.get(k) for x in rag) for k in _METRICS},
                    "overall": self._mean(x.get("overall") for x in rag)},
            "plain": {k: self._mean(x.get(k) for x in plain) for k in _METRICS},
            "insufficient_rate": self._mean(float(x["insufficient"]) for x in rag) if rag else None,
            "validator_ok_rate": self._mean(float(x["validator_ok"]) for x in rag) if rag else None,
            "support_win_rate": round(sum(a > b for a, b in both) / len(both), 4) if both else None,
            "mean_support_gain": self._mean(a - b for a, b in both),
            "n_compared": len(both),
        }

    def to_csv(self, path):
        with self._lock:
            rows = [flatten(r) for r in self.records]
        cols = list(dict.fromkeys(k for r in rows for k in r))
        with open(path, "w", newline="", encoding="utf-8-sig") as f:   # utf-8-sig: 엑셀에서 한글 깨짐 방지
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            w.writerows(rows)
        return path


eval_log = globals().get("eval_log") or EvalLog(EVAL_CONFIG["log_path"])   # 셀을 다시 실행해도 기록 유지

In [ ]:
# ──────────────── 요청 모델·설정 ────────────────
class AskInput(BaseModel):
    question: str
    compare: bool = False        # True면 근거 없는 일반 LLM 답변도 함께 생성

ASK_CONFIG = {"related_top_k": 4, "related_min_score": 0.2}   # CELL 14와 같은 값
SNIPPET_CHARS = 220              # 결과창에 보여 줄 참조 구절 길이
MAX_QUESTION_CHARS = 2000

HTML = r"""
<html lang="ko">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1">
        <title>VeriFive - 근거 기반 경제 뉴스 챗봇</title>
        <style>
            :root {
                --bg: #eff7f3;
                --panel: #f7fbf9;
                --ink: #103a2d;
                --muted: #718079;
                --green: #006b4f;
                --line: #d5e5dd;
                --mint: #d9efe5
            }

            * { box-sizing: border-box }

            body {
                margin: 0;
                background: var(--bg);
                color: var(--ink);
                font: 14px/1.7 Arial,'Noto Sans KR','Malgun Gothic',sans-serif
            }

            button, input, select, textarea { font: inherit; color: inherit }
            button { cursor: pointer }
            button:disabled { cursor: wait; opacity: .6 }

            button, select, input[type="text"] {
                border: 1px solid var(--line);
                border-radius: 6px;
                background: var(--panel);
                padding: 9px 13px
            }

            button.primary {
                background: var(--green);
                border-color: var(--green);
                color: white;
                font-weight: 700
            }

            header {
                height: 74px;
                background: var(--panel);
                border-bottom: 1px solid var(--line);
                display: flex; align-items: center; justify-content: space-between;
                padding: 0 30px
            }

            .brand { display: flex; gap: 10px; align-items: center }
            .logo {
                background: #092e21; color: white;
                font: 700 27px Georgia,serif;
                width: 37px; height: 37px; border-radius: 8px;
                display: grid; place-items: center
            }

            .brand strong { display: block; font-size: 19px; line-height: 1.2 }
            .brand small { color: var(--muted); font-size: 11px }

            .shell { display: flex; min-height: calc(100vh - 118px) }

            aside {
                width: 230px; flex-shrink: 0;
                background: var(--panel); border-right: 1px solid var(--line);
                padding: 25px 16px; display: flex; flex-direction: column;
                justify-content: space-between
            }

            .nav { display: grid; gap: 7px }
            .nav button {
                width: 100%; display: flex; align-items: center; gap: 10px;
                border: 0; background: none; text-align: left;
                color: #65746d; padding: 12px; border-radius: 6px;
            }
            .nav button.active { background: var(--mint); color: var(--ink); font-weight: 700 }
            .nav .count { margin-left: auto }

            main {
                width: min(1100px, 100%);
                padding: 46px 58px 65px;
                margin: 0 auto;
                display: flex;
                flex-direction: column;
                justify-content: center;
                min-height: calc(100vh - 74px);
            }
            .pre { white-space: pre-wrap }

            .page { display: none; width: 100%; }
            .page.active { display: block; }

            .hero h1 {
                font: 38px/1.35 Georgia,'Noto Serif KR','Batang',serif;
                margin: 11px 0 15px; letter-spacing: -1.5px; font-weight: 500
            }

            .hero p { color: var(--muted); margin: 0 0 29px; font-size: 13px }

            .question-panel {
                padding: 18px 0 20px;
                border-top: 1px solid var(--line);
                border-bottom: 1px solid var(--line)
            }

            textarea, input[type="text"] {
                display: block; width: 100%;
                padding: 13px; border: 1px solid #7d8d85; border-radius: 6px;
                background: var(--panel); margin-top: 8px
            }
            textarea { min-height: 125px; resize: vertical; }

            .under { display: flex; justify-content: flex-end; align-items: center; margin: 7px 0 20px }
            .under span { color: var(--muted); font-size: 11px }

            .submit-row { display: flex; justify-content: space-between; align-items: center; margin-top: 10px }
            .check { display: flex; gap: 8px; align-items: center }

            .guide-box {
                background: var(--panel);
                border: 1px solid var(--line);
                border-radius: 8px;
                padding: 20px;
                margin-bottom: 20px;
            }
            .guide-box h3 { margin-top: 0; color: var(--green); }
            code { background: #e2ede8; padding: 2px 6px; border-radius: 4px; font-size: 13px; }

            dialog {
                width: min(850px, calc(100% - 30px));
                background: var(--panel);
                border: 1px solid var(--line);
                border-radius: 12px;
                padding: 26px;
                color: var(--ink);
                box-shadow: 0 20px 90px #103a2d30;
            }
            dialog::backdrop { background: #09291f66 }

            .metrics { display: grid; grid-template-columns: repeat(4,1fr); border-block:1px solid var(--line); margin: 22px 0 }
            .metric { padding: 13px; border-right: 1px solid var(--line) }
            .metric:last-child { border: 0 }
            .metric span { display: block; font-size: 11px; color: var(--muted) }
            .metric strong { font-size: 18px }
            .answer-grid { display: grid; grid-template-columns: 1fr; gap: 26px }

            .spinner {
                display: inline-block; width: 15px; height: 15px;
                border: 2px solid var(--line); border-top-color: var(--green);
                border-radius: 50%; animation: spin .7s linear infinite;
                vertical-align: middle; margin-right: 8px
            }
            @keyframes spin { to { transform: rotate(360deg) } }

            .progress-box { margin-top: 22px }
            .progress-head { display: flex; justify-content: space-between; align-items: baseline; margin-bottom: 8px }
            .progress-head strong { font-size: 22px; color: var(--green) }
            .progress-track { height: 12px; background: var(--mint); border-radius: 999px; overflow: hidden }
            .progress-fill { height: 100%; width: 0; background: var(--green); border-radius: 999px; transition: width .5s ease }
            .progress-fill.error { background: #b3261e }

            .toast {
                position: fixed; right: 24px; bottom: 24px; z-index: 1000;
                background: var(--green); color: #fff; padding: 14px 20px; border-radius: 8px;
                box-shadow: 0 8px 30px #103a2d40; font-weight: 700;
                opacity: 0; transform: translateY(12px); transition: .3s; pointer-events: none
            }
            .toast.show { opacity: 1; transform: none }
            .toast.error { background: #b3261e }

            [hidden] { display: none !important }
            .dlg-head { display: flex; justify-content: space-between; align-items: center; margin-bottom: 15px }
            .dlg-head h2 { margin: 0 }
            .state-box { text-align: center; padding: 40px 0 }
            .state-box.err { color: #b3261e; font-weight: 700; background: #fdecea; border-radius: 8px; padding: 18px }
            .answer-box { background: #fff; border: 1px solid var(--line); border-radius: 8px; padding: 16px 18px }
            .ans-line { margin: 0 0 10px }
            .ans-line:last-child { margin-bottom: 0 }
            .cite {
                display: inline-block; margin-left: 4px; padding: 0 7px;
                font-size: 11px; font-weight: 700; line-height: 20px;
                color: var(--green); background: var(--mint);
                border: 1px solid var(--green); border-radius: 999px
            }
            .notice { background: #fff6e0; border: 1px solid #f0d58a; color: #6b4e00; border-radius: 8px; padding: 12px 14px; margin: 12px 0 }
            .notice.warn { background: #fdecea; border-color: #f2b8b5; color: #8c1d18 }
            .src-item { border: 1px solid var(--line); border-radius: 8px; padding: 12px 14px; margin-bottom: 10px; background: #fff }
            .src-item.flash { box-shadow: 0 0 0 3px #00b37a66 }
            .src-head { display: flex; gap: 8px; align-items: baseline }
            .badge { flex-shrink: 0; font-size: 11px; font-weight: 700; color: #fff; background: var(--green); border-radius: 4px; padding: 1px 7px }
            .badge.gray { background: #718079 }
            .src-title { font-weight: 700; color: var(--ink); word-break: break-all }
            .src-meta { color: var(--muted); font-size: 12px; margin-top: 2px }
            details { margin-top: 6px }
            summary { cursor: pointer; font-size: 12px; color: var(--green) }
            .snippet { margin: 6px 0 0; padding: 6px 10px; border-left: 3px solid var(--line); color: #4b5a53; font-size: 12px; white-space: pre-wrap }

            .notice.info { background: #eaf4ef; border-color: var(--line); color: var(--ink) }
            .progress-box { margin-top: 0; text-align: left }
        </style>
    </head>
    <body>
        <header>
            <div class="brand">
                <span class="logo">V</span>
                <div>
                    <strong>VeriFive</strong>
                    <small>근거 기반 뉴스 챗봇</small>
                </div>
            </div>
        </header>

        <div class="shell">
            <aside>
                <nav class="nav">
                    <button id="navQa" class="active" onclick="switchTab('qa')">▤ 질의응답</button>
                    <button id="navGuide" onclick="switchTab('guide')">📘 연결가이드</button>
                    <button id="navEval" onclick="switchTab('evalPage')">📊 평가</button>
                </nav>
            </aside>
            <main>
                <!-- 1. 질의응답 화면 -->
                <section id="qa" class="page active">
                    <div class="hero">
                        <div class="eyebrow">EVIDENCE CONSOLE</div>
                        <h1>궁금한 내용을 입력하면,<br>네이버 뉴스에서 근거를 찾아 답합니다.</h1>
                        <p>입력한 문장에서 키워드를 뽑아 관련 기사를 수집하고, 그 기사들을 근거로 답변과 출처를 보여줍니다.</p>
                    </div>

                    <form id="questionForm" class="question-panel">
                        <label for="question">확인하고 싶은 내용 (질문 또는 주장)</label>
                        <textarea id="question" maxlength="2000" placeholder="예) 컨닝 전문 조직이 있어?" required></textarea>

                        <div class="under">
                            <span id="charCount">0 / 2,000</span>
                        </div>

                        <div class="submit-row">
                            <label class="check">
                                <input id="compare" type="checkbox" checked>일반 LLM과 비교
                            </label>
                            <button class="primary" id="generateButton" type="submit">➤ 근거 수집 및 답변 생성</button>
                        </div>
                    </form>
                </section>

                <!-- 2. 연결가이드 화면 -->
                <section id="guide" class="page">
                    <div class="hero">
                        <div class="eyebrow">BACKEND PIPELINE GUIDE</div>
                        <h1>입력 문장에서 답변까지<br>처리되는 순서</h1>
                        <p>질문 하나가 들어오면 서버에서 아래 순서로 처리하고, 진행률을 결과창에 보여줍니다.</p>
                    </div>

                    <div class="guide-box">
                        <h3>1. 근거 수집 (`/api/ask`)</h3>
                        <p>입력 문장에서 <strong>키워드 추출 ➔ 네이버 뉴스 검색 ➔ 관련 기사 수집·정제 ➔ 청킹 ➔ 임베딩 ➔ 벡터 DB(FAISS) 추가</strong>를 수행합니다.</p>
                    </div>

                    <div class="guide-box">
                        <h3>2. 답변 생성</h3>
                        <p>이번 질문으로 수집한 근거 기사 청크만 검색해 <code>RAGGenerator.create_response</code>로 답변을 만들고, <code>[근거 N]</code> 출처와 자동 검사 결과를 결과창에 표시합니다.</p>
                    </div>
                </section>

                <!-- 3. 평가 화면 -->
                <section id="evalPage" class="page">
                    <div class="hero">
                        <div class="eyebrow">EVALUATION</div>
                        <h1>일반 LLM과 근거 기반 답변,<br>수치로 비교하기</h1>
                        <p>질문을 한 줄에 하나씩 입력하세요(최대 2000개). 정답 문장이 있으면 질문 뒤에 <code>|||</code>로 이어 붙이면 정답 유사도도 계산합니다.</p>
                    </div>
                    <form id="evalForm" class="question-panel">
                        <textarea id="evalInput" placeholder="질문 1&#10;질문 2 ||| 정답 문장"></textarea>
                        <div class="submit-row">
                            <span id="evalStatus" style="color:var(--muted);font-size:12px"></span>
                            <button class="primary" id="evalButton" type="submit">▶ 평가 실행</button>
                        </div>
                    </form>
                    <div id="evalSummary" hidden style="margin-top:22px"></div>
                </section>
            </main>
        </div>

        <!-- 결과 전용 모달 팝업창 -->
        <dialog id="resultDialog">
            <div class="dlg-head">
                <h2>분석 결과 및 출처</h2>
                <button type="button" id="closeDialogBtn">닫기 ✕</button>
            </div>

            <div id="loadingState" class="state-box">
                <div class="progress-box">
                    <div class="progress-head">
                        <span id="progressStage">준비 중</span>
                        <strong id="progressPct">0%</strong>
                    </div>
                    <div class="progress-track"><div id="progressBar" class="progress-fill"></div></div>
                </div>
            </div>
            <div id="errorState" class="state-box err" hidden></div>

            <div id="result" hidden>
                <div id="kwNote" class="notice info" hidden></div>

                <div class="metrics">
                    <div class="metric"><span>인용된 출처</span><strong id="metricSourceCount">0</strong></div>
                    <div class="metric"><span>참조된 구절</span><strong id="metricChunkCount">0개</strong></div>
                    <div class="metric"><span>인용 검사</span><strong id="citeState">-</strong></div>
                    <div class="metric"><span>생성 시간</span><strong id="modelState">-</strong></div>
                </div>

                <div id="insufficientNote" class="notice" hidden></div>

                <h3>근거 기반 답변</h3>
                <div id="answerBox" class="answer-box"></div>
                <div id="checkErrors" class="notice warn" hidden></div>

                <div id="plainBox" hidden style="margin-top: 20px;">
                    <h3>일반 LLM 답변 (근거 없음)</h3>
                    <div class="answer-box"><p id="plainText" class="pre" style="margin:0"></p></div>
                </div>

                <div id="evalBox" hidden style="margin-top: 20px;">
                    <h3>평가 지표</h3>
                    <div id="evalTable" class="answer-box"></div>
                </div>

                <div style="margin-top: 20px; border-top: 1px solid var(--line); padding-top: 15px;">
                    <h3>참조 출처 목록</h3>
                    <div id="sourceList" style="margin-top: 8px; font-size: 13px;"></div>
                </div>
            </div>
        </dialog>

        <script>
            const $ = id => document.getElementById(id);

            const TABS = { qa: 'navQa', guide: 'navGuide', evalPage: 'navEval' };
            function switchTab(tabName) {
                Object.keys(TABS).forEach(name => {
                    $(name).classList.toggle('active', name === tabName);
                    $(TABS[name]).classList.toggle('active', name === tabName);
                });
            }

            $('question').addEventListener('input', () => {
                $('charCount').textContent = $('question').value.length.toLocaleString() + ' / 2,000';
            });
            $('closeDialogBtn').addEventListener('click', () => $('resultDialog').close());

            // 서버가 준 값은 모두 textContent로만 넣어 XSS를 막습니다.
            function el(tag, cls, text) {
                const e = document.createElement(tag);
                if (cls) e.className = cls;
                if (text !== undefined && text !== null) e.textContent = text;
                return e;
            }
            function safeUrl(u) {
                u = u || '';
                return (u.startsWith('http://') || u.startsWith('https://')) ? u : null;
            }
            function jumpToSource(n) {
                const t = $('src-' + n);
                if (!t) return;
                t.scrollIntoView({ behavior: 'smooth', block: 'center' });
                t.classList.add('flash');
                setTimeout(() => t.classList.remove('flash'), 1600);
            }
            function setProgress(pct, stage) {
                $('progressBar').style.width = pct + '%';
                $('progressPct').textContent = pct + '%';
                $('progressStage').textContent = stage;
            }

            function renderSourceItem(s) {
                const item = el('div', 'src-item');
                item.id = 'src-' + s.number;
                const head = el('div', 'src-head');
                head.appendChild(el('span', 'badge', '근거 ' + s.number));
                const url = safeUrl(s.url);
                if (url) {
                    const a = el('a', 'src-title', s.title || '(제목 없음)');
                    a.href = url; a.target = '_blank'; a.rel = 'noopener noreferrer';
                    head.appendChild(a);
                } else {
                    head.appendChild(el('span', 'src-title', s.title || '(제목 없음)'));
                }
                item.appendChild(head);
                item.appendChild(el('div', 'src-meta', [s.press, s.published_at].filter(Boolean).join(' · ')));
                if (s.snippets && s.snippets.length) {
                    const d = el('details');
                    d.appendChild(el('summary', null, '참조 구절 ' + s.snippets.length + '개 보기'));
                    s.snippets.forEach(t => d.appendChild(el('blockquote', 'snippet', t)));
                    item.appendChild(d);
                }
                return item;
            }

            // ── 평가 지표 표시 (label, 키, 형식, RAG 전용 여부)
            const EVAL_ROWS = [
                ['근거 충족률 (답변 문장이 근거와 일치)', 'support_rate', 'pct', false],
                ['평균 근거 유사도', 'mean_support', 'num', false],
                ['질문 관련도', 'answer_relevancy', 'num', false],
                ['인용 정확도', 'citation_precision', 'pct', true],
                ['인용 포함 비율', 'citation_coverage', 'pct', true],
                ['정답 유사도 (정답 입력 시)', 'ref_sim', 'num', false],
                ['생성 시간(초)', 'generation_ms', 'sec', false],
            ];
            function fmtMetric(v, kind) {
                if (v === null || v === undefined) return '-';
                if (kind === 'pct') return (v * 100).toFixed(0) + '%';
                if (kind === 'sec') return (v / 1000).toFixed(1);
                return Number(v).toFixed(2);
            }
            function makeTable(headers, rows) {
                const t = el('table');
                t.style.cssText = 'width:100%;border-collapse:collapse;font-size:13px';
                const addRow = (cells, tag) => {
                    const tr = el('tr');
                    cells.forEach((c, i) => {
                        const cell = el(tag, null, c);
                        cell.style.cssText = 'padding:6px 8px;border-bottom:1px solid var(--line);text-align:' + (i ? 'right' : 'left');
                        tr.appendChild(cell);
                    });
                    t.appendChild(tr);
                };
                addRow(headers, 'th');
                rows.forEach(r => addRow(r, 'td'));
                return t;
            }
            function metricRows(rag, plain, overall) {
                const rows = EVAL_ROWS.map(([label, key, kind, ragOnly]) =>
                    [label, fmtMetric(rag ? rag[key] : null, kind), ragOnly ? '-' : fmtMetric(plain ? plain[key] : null, kind)]);
                rows.push(['RAG 종합점수 (가중 평균)', fmtMetric(overall, 'num'), '-']);
                return rows;
            }
            function renderEval(ev) {
                const box = $('evalBox');
                if (!ev) { box.hidden = true; return; }
                box.hidden = false;
                const t = $('evalTable');
                t.replaceChildren();
                t.appendChild(makeTable(['지표', '근거 기반(RAG)', '일반 LLM'], metricRows(ev.rag, ev.plain, ev.rag.overall)));
                                if (ev.rag.insufficient) t.appendChild(el('p', 'src-meta', '⚠ 근거 부족으로 RAG 답변 지표를 계산하지 않았습니다.'));
                const c = ev.comparison || {};
                if (c.support_gain !== null && c.support_gain !== undefined) {
                    t.appendChild(el('p', 'src-meta', '근거 충족률 차이: ' + (c.support_gain * 100).toFixed(0) + '%p · 일반 LLM 답변 중 근거로 확인되지 않은 문장: '
                        + fmtMetric(c.plain_unsupported_rate, 'pct')));
                }
                const bad = (ev.rag.unsupported_claims || []);
                if (bad.length) t.appendChild(el('p', 'src-meta', '근거 미확인 문장(RAG): ' + bad.join(' / ')));
            }

            function renderResult(data) {
                const sources = data.sources || [];
                const v = data.validation || {};
                const errs = v.errors || [];

                $('sourceList').previousElementSibling.textContent =
                    data.insufficient ? '검색된 기사 (답변에는 사용되지 않음)' : '참조 출처 목록';

                // 검색 키워드·수집 현황
                const kw = $('kwNote');
                if ((data.keywords || []).length) {
                    kw.textContent = '검색 키워드: ' + data.keywords.join(', ') + ' · 수집한 관련 기사 ' + (data.related_found || 0) + '건';
                    kw.hidden = false;
                } else {
                    kw.hidden = true;
                }

                // 요약 지표
$('metricSourceCount').textContent = data.insufficient ? 0 : (data.source_count ?? sources.length);
$('metricChunkCount').textContent = (data.insufficient ? 0 : (data.chunk_count ?? 0)) + '개';
                $('citeState').textContent = data.insufficient ? '해당 없음' : (v.ok ? '통과' : ('확인 필요 (' + errs.length + ')'));
                $('modelState').textContent = (data.generation_ms != null) ? (data.generation_ms / 1000).toFixed(1) + '초' : '-';

                // 근거 부족 안내
                const note = $('insufficientNote');
                note.hidden = !data.insufficient;
                if (data.insufficient) note.textContent = data.insufficient_reason || '수집한 근거 기사만으로는 답할 수 없습니다.';

                // 답변 (문장 + [근거 N] 배지)
                const box = $('answerBox');
                box.replaceChildren();
                (data.answer_lines || []).forEach(line => {
                    const p = el('p', 'ans-line');
                    p.appendChild(document.createTextNode(line.text + ' '));
                    (line.cites || []).forEach(n => {
                        const b = el('button', 'cite', '근거 ' + n);
                        b.type = 'button';
                        b.addEventListener('click', () => jumpToSource(n));
                        p.appendChild(b);
                    });
                    box.appendChild(p);
                });
                if (!box.childNodes.length) box.textContent = data.answer || '';

                // 자동 검사 경고
                const ce = $('checkErrors');
                ce.hidden = data.insufficient || !errs.length;
                if (!ce.hidden) ce.textContent = '⚠ 자동 검사 경고: ' + errs.join(' / ');

                // 일반 LLM 비교 답변
                $('plainBox').hidden = !data.plain_answer;
                if (data.plain_answer) $('plainText').textContent = data.plain_answer;

                renderEval(data.evaluation);

                // 출처 목록
                const list = $('sourceList');
                list.replaceChildren();
                if (sources.length) {
                    sources.forEach(s => list.appendChild(renderSourceItem(s)));
                } else {
                    list.textContent = '인용된 출처가 없습니다.';
                    list.style.color = 'var(--muted)';
                }
            }

            // 작업 진행률 조회 (서버가 근거 수집→임베딩→답변 생성을 순서대로 처리)
            async function pollAsk(jobId, onProgress) {
                let failures = 0;
                while (true) {
                    await new Promise(resolve => setTimeout(resolve, 700));
                    try {
                        const res = await fetch('/api/ask/status/' + jobId);
                        const job = await res.json();
                        if (!res.ok) throw new Error(job.detail || '상태 조회 실패');
                        failures = 0;
                        setProgress(job.percent, job.stage);
                        if (onProgress) onProgress(job.percent, job.stage);
                        if (job.status === 'done') return job.result;
                        if (job.status === 'error') throw new Error(job.error || '처리 중 오류 발생');
                    } catch (error) {
                        if (error.message && !(error instanceof TypeError)) throw error;
                        if (++failures >= 5) throw new Error('서버와 통신할 수 없습니다.');
                    }
                }
            }

            let asking = false;
            $('questionForm').addEventListener('submit', async event => {
                event.preventDefault();
                if (asking) return;
                const question = $('question').value.trim();
                if (!question) return;

                asking = true;
                const btn = $('generateButton');
                btn.disabled = true;
                $('result').hidden = true;
                $('errorState').hidden = true;
                $('loadingState').hidden = false;
                setProgress(0, '요청 전송 중');
                if (!$('resultDialog').open) $('resultDialog').showModal();

                try {
                    const res = await fetch('/api/ask', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ question: question, compare: $('compare').checked })
                    });
                    let data = null;
                    try { data = await res.json(); } catch (_) { /* JSON이 아닌 응답 */ }
                    if (!res.ok) throw new Error((data && data.detail) || ('서버 오류 (HTTP ' + res.status + ')'));

                    const result = await pollAsk(data.job_id);
                    renderResult(result);
                } catch (error) {
                    console.error(error);
                    $('loadingState').hidden = true;
                    $('errorState').textContent = '❌ ' + (error.message || '오류가 발생했습니다.');
                    $('errorState').hidden = false;
                } finally {
                    asking = false;
                    btn.disabled = false;
                }
                $('sourceList').previousElementSibling.textContent =
                data.insufficient ? '검색된 기사 (답변에는 사용되지 않음)' : '참조 출처 목록';
                $('loadingState').hidden = true;
                $('result').hidden = false;
            });
              function renderBatch(r) {
    const box = $('evalSummary');
    box.replaceChildren();
    box.hidden = false;
    const s = r.summary || {}, rag = s.rag || {}, plain = s.plain || {};

    box.appendChild(el('h3', null, '평가 결과 (질문 ' + (s.n || 0) + '개)'));
    const rows = EVAL_ROWS.map(([label, key, kind, ragOnly]) =>
        [label, fmtMetric(rag[key], kind), ragOnly ? '-' : fmtMetric(plain[key], kind)]);
    rows.push(['RAG 종합점수 (가중 평균)', fmtMetric(rag.overall, 'num'), '-']);
    box.appendChild(makeTable(['지표', '근거 기반(RAG)', '일반 LLM'], rows));

    box.appendChild(el('p', 'src-meta',
        '근거 부족 비율: ' + fmtMetric(s.insufficient_rate, 'pct') +
        ' · 자동 검사 통과율: ' + fmtMetric(s.validator_ok_rate, 'pct') +
        ' · RAG가 더 높은 질문 비율: ' + fmtMetric(s.support_win_rate, 'pct') +
        ' (비교 ' + (s.n_compared || 0) + '건)'));

    const detail = (r.rows || []).map(x => x.error
        ? [x.question, '오류', '-', '-', x.error]
        : [x.question, x.status, fmtMetric(x.rag_support, 'pct'), fmtMetric(x.plain_support, 'pct'), x.reason || '']);
    box.appendChild(el('h3', null, '질문별 결과'));
    const wrap = el('div');
    wrap.style.overflowX = 'auto';
    wrap.appendChild(makeTable(['질문', '상태', 'RAG 충족률', '일반 LLM 충족률', '비고'], detail));
    box.appendChild(wrap);

    if (r.csv) {
        const id = r.csv.replace('eval_batch_', '').replace('.csv', '');
        const a = el('a', null, 'CSV 내려받기');
        a.href = '/api/eval/csv/' + id;
        const p = el('p');
        p.appendChild(a);
        box.appendChild(p);
    }
}

let evaluating = false;
$('evalForm').addEventListener('submit', async event => {
    event.preventDefault();
    if (evaluating) return;
    const items = $('evalInput').value.split(String.fromCharCode(10))
        .map(l => l.trim()).filter(Boolean)
        .map(l => {
            const i = l.indexOf('|||');
            return i < 0 ? { question: l }
                         : { question: l.slice(0, i).trim(), reference: l.slice(i + 3).trim() };
        })
        .filter(it => it.question);
    if (!items.length) { $('evalStatus').textContent = '질문을 입력해주세요.'; return; }

    evaluating = true;
    $('evalButton').disabled = true;
    $('evalSummary').hidden = true;
    $('evalStatus').textContent = '평가 요청 중...';
    try {
        const res = await fetch('/api/eval/batch', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ items: items })
        });
        const data = await res.json();
        if (!res.ok) throw new Error(data.detail || ('서버 오류 (HTTP ' + res.status + ')'));
        const result = await pollAsk(data.job_id, (pct, stage) => {
            $('evalStatus').textContent = pct + '% · ' + (stage || '');
        });
        $('evalStatus').textContent = '완료';
        renderBatch(result);
    } catch (error) {
        console.error(error);
        $('evalStatus').textContent = '❌ ' + (error.message || '오류가 발생했습니다.');
    } finally {
        evaluating = false;
        $('evalButton').disabled = false;
    }
});

        </script>
    </body>
</html>
"""

# ──────────────── 작업 큐: GPU를 쓰는 모든 처리는 모델을 적재한 메인 스레드에서 순서대로 (셀 맨 아래 참고) ────────────────
jobs = {}
query_cache = globals().get("query_cache") or {}   # {질문 ID: 수집 결과 요약} 같은 질문은 네이버 재수집 생략
if "related_articles" not in globals():
    related_articles = []

_job_queue = queue.Queue()


def _set(job_id, **kw):
    jobs[job_id].update(kw)


def _get_generator():
    """CELL 19에서 만든 generator를 재사용하고, 없으면 새로 만든다(모델은 첫 생성 때 적재)."""
    global generator
    if globals().get("generator") is None:
        generator = RAGGenerator()
    return generator


def _fmt_date(value):
    value = str(value or "")
    return value[:16].replace("T", " ") if value and value != "미확인" else ""


# ──────────────── 근거 수집 파이프라인 연결 (cell_pipeline.py의 EvidencePipeline 조립) ────────────────
def _build_pipeline():
    # 청킹 임베딩 모델 생성
    global model
    model = create_model()
    # 앞 셀에서 만든 repository(누적 벡터 DB)가 있으면 이어서 쓰고, 없으면 SAVE_DIR에서 복원/새로 생성
    repo = globals().get("repository") or VectorRepository(SAVE_DIR)
    chunk_cfg = globals().get("CONFIG")
    if not isinstance(chunk_cfg, ChunkConfig):
        chunk_cfg = ChunkConfig()
    return EvidencePipeline(
        keyword_extractor=KeywordExtractor(keyword_analyzer, RELATED_NEWS_CONFIG["max_keywords"]),
        source=NaverNewsSource(),      # 다른 수집원은 ArticleSource만 구현해 교체
        chunker=ArticleChunker(model, config=chunk_cfg),
        embedder=ChunkEmbedder(model, model_name = "BAAI/bge-m3", batch_size=16),
        repository=repo,
        cleaners=[],                                       # 필요하면 정제 함수 추가
    )

pipeline = _build_pipeline()
repository = pipeline.repository
vector_store = repository.store                            # 다른 셀이 쓰는 전역과 동기화


def _collect_evidence(job_id, text, qid):
    """UI 쪽 역할: 캐시 확인, 진행률 표시, 전역 상태 동기화. 실제 처리는 pipeline.run()."""
    global vector_store
    if qid in query_cache:                                          # 같은 질문을 다시 한 경우
        _set(job_id, percent=50, stage="이전에 수집한 근거 기사를 재사용합니다")
        return query_cache[qid]

    def progress(percent, stage=None):
        _set(job_id, percent=percent, **({"stage": stage} if stage else {}))

    info = pipeline.run(text, qid, on_progress=progress)

    vector_store = pipeline.repository.store                        # 누적 벡터 DB 동기화
    related_articles.extend(info.pop("_used_articles", []))
    query_cache[qid] = info
    return info


def _search_evidence(text, article_ids, top_k, min_score):
    """이번 질문에 연결된 기사(article_ids)의 청크만 검색한다.
    파이프라인은 청크에 질문 ID를 달지 않으므로 FAISS 전체 검색 후 article_id로 거른다."""
    store = pipeline.repository.store
    if store is None or store["index"].ntotal == 0 or not article_ids:
        return []
    q = model.encode([text], convert_to_numpy=True, show_progress_bar=False).astype("float32")
    faiss.normalize_L2(q)                                           # 저장된 벡터와 같은 L2 정규화

    scores, idxs = store["index"].search(q, store["index"].ntotal)
    allowed, out = set(article_ids), []
    for score, i in zip(scores[0], idxs[0]):
        if i < 0:
            continue
        if score < min_score:                                       # 내적 점수는 내림차순이라 여기서 중단
            break
        chunk = store["chunks"][i]
        if chunk.get("article_id") not in allowed:
            continue
        meta = chunk.get("metadata") or {}
        out.append({**meta, **chunk, "score": float(score),
                    "retrieval_group": "related",
                    "source_type": "related"})
        if len(out) >= top_k:
            break
    return out


def _answer_question(job_id, text, compare):
    """근거 수집 → 검색 → RAG 답변(+일반 LLM 답변). (rag, retrieved, plain, info) 반환."""
    qid = " ".join(text.split()).casefold()

    info = _collect_evidence(job_id, text, qid)

    # 이번 질문으로 수집한 근거 기사 청크만 검색
    _set(job_id, percent=80, stage="관련 구절 검색 중")
    retrieved = _search_evidence(
        text, info.get("article_ids", []),
        top_k=ASK_CONFIG["related_top_k"], min_score=ASK_CONFIG["related_min_score"])

    # 근거 기반 답변 (근거가 없으면 모델을 호출하지 않고 '근거 부족' 반환)
    _set(job_id, percent=85, stage="근거 기반 답변 생성 중 (첫 실행은 모델 적재로 오래 걸릴 수 있습니다)")
    gen = _get_generator()
    rag = gen.create_response(text, retrieved)
    plain = None
    if compare:
        _set(job_id, percent=95, stage="일반 LLM 답변 생성 중")
        plain = gen.create_plain_response(text)
    return rag, retrieved, plain, info

def _evaluate(text, rag, retrieved, plain, reference=None, raise_errors=False):
    """평가가 실패해도 답변 표시는 막지 않는다. 성공하면 누적 기록(eval_log)에도 남긴다."""
    try:
        ev = evaluate_case(text, rag, retrieved, plain, model, reference=reference)
        eval_log.add(ev)
        return ev
    except Exception:
        traceback.print_exc()
        if raise_errors:                      # 배치 평가에서는 실제 오류 내용을 행에 표시
            raise
        return None


def _run_job(job_id, text, compare):
    started = time.perf_counter()
    _set(job_id, status="running", percent=2, stage="처리 시작")
    rag, retrieved, plain, info = _answer_question(job_id, text, compare)
    result = _to_ui_payload(rag, retrieved, plain, (time.perf_counter() - started) * 1000, info)
    result["evaluation"] = _evaluate(text, rag, retrieved, plain)
    _set(job_id, status="done", percent=100, stage="완료", result=result)


MAX_EVAL_BATCH = 2000


def _batch_row(ev, info):
    rag = ev["rag"]
    if rag.get("insufficient"):
        status = "근거 부족"
        if rag.get("n_chunks", 0) == 0 and info.get("found", 0) == 0:
            reason = "관련 기사를 수집하지 못함"
        elif rag.get("n_chunks", 0) == 0:
            reason = f"기사 {info.get('found', 0)}건 수집, 유사도 기준({ASK_CONFIG['related_min_score']}) 미달"
        else:
            reason = "모델이 근거만으로 답할 수 없다고 판단"
    elif rag.get("support_rate") is None:
        status, reason = "채점 불가", "답변에서 채점할 문장을 찾지 못함"
    else:
        status, reason = "정상", ""
    return {"question": ev["question"], "status": status, "reason": reason,
            "rag_support": rag["support_rate"],
            "plain_support": (ev["plain"] or {}).get("support_rate"),
            "citation_precision": rag.get("citation_precision"),
            "overall": rag.get("overall")}


def _run_batch(job_id, items):
    """질문 여러 개를 차례로 처리해 평균 지표를 만든다. 질문마다 임시 작업 기록을 만들어 진행 표시를 분리한다."""
    n = len(items)
    batch_log, rows = EvalLog(None), []
    _set(job_id, status="running", percent=1, stage=f"평가 시작 (0/{n})")
    for i, item in enumerate(items, 1):
        q = " ".join(str(item.get("question", "")).split())
        ref = " ".join(str(item.get("reference") or "").split()) or None
        _set(job_id, percent=int(100 * (i - 1) / n), stage=f"평가 중 ({i}/{n}) {q[:30]}")
        tmp = "b" + uuid.uuid4().hex[:10]
        jobs[tmp] = {"status": "running", "percent": 0, "stage": "", "error": None, "result": None}
        try:
            rag, retrieved, plain, info = _answer_question(tmp, q, True)
            ev = _evaluate(q, rag, retrieved, plain, reference=ref, raise_errors=True)
            batch_log.add(ev)
            rows.append(_batch_row(ev, info))
        except Exception as e:
            traceback.print_exc()
            rows.append({"question": q, "error": f"{type(e).__name__}: {e}"})
        finally:
            jobs.pop(tmp, None)
    csv_path = None
    if batch_log.records:
        csv_path = batch_log.to_csv(f"eval_batch_{job_id}.csv")
    _set(job_id, status="done", percent=100, stage="완료",
         result={"summary": batch_log.summary(), "rows": rows, "csv": csv_path})


def _worker(q):
    while True:
        task = q.get()
        if task is None:
            break
        job_id, payload, compare = task
        try:
            if isinstance(payload, dict):                 # 평가 묶음 작업
                _run_batch(job_id, payload["items"])
            else:
                _run_job(job_id, payload, compare)
        except Exception as e:
            traceback.print_exc()
            failed_at = jobs[job_id].get("stage") or ""
            _set(job_id, status="error", stage="실패", error=f"[{failed_at}] {type(e).__name__}: {e}")


# ──────────────── RAG 결과 → UI 응답 형식 ────────────────
def _answer_lines(answer):
    """답변을 문장 단위로 나누고, 끝의 [근거 N] 묶음을 번호 목록으로 분리한다(CELL 15의 정규식 재사용)."""
    out = []
    for line in (answer or "").splitlines():
        line = line.strip()
        if not line:
            continue
        m = CITATION_BLOCK.search(line)
        if m:
            cites = list(dict.fromkeys(int(n) for n in CITATION.findall(m.group(0))))
            out.append({"text": line[:m.start()].strip(), "cites": cites})
        else:
            out.append({"text": line, "cites": []})
    return out


def _to_ui_payload(rag, retrieved, plain, elapsed_ms, info):
    text_by_chunk = {c.get("chunk_id"): c.get("text", "") for c in retrieved}

    def snippet(cid):
        t = text_by_chunk[cid].strip()
        return t[:SNIPPET_CHARS] + ("…" if len(t) > SNIPPET_CHARS else "")

    sources = [{
        "number": s.get("number"),
        "title": s.get("title") or "미확인",
        "press": s.get("press") if s.get("press") != "미확인" else "",
        "published_at": _fmt_date(s.get("published_at")),
        "url": s.get("url") or "",
        "snippets": [snippet(cid) for cid in s.get("chunk_ids", []) if cid in text_by_chunk],
    } for s in rag.get("sources", [])]

    gen = rag.get("generation") or {}
    val = rag.get("validation") or {}
    insufficient = bool(rag.get("insufficient_evidence"))

    reason = ""
    if insufficient:
        if not retrieved and info.get("found", 0) == 0:
            reason = ("네이버 뉴스 검색에서 근거로 쓸 기사를 찾지 못했습니다. "
                      "키워드: " + ",".join(info.get("keywords", [])) + " — 표현을 바꾸거나 더 구체적으로 입력해 보세요.")
        elif not retrieved:
            reason = (f"관련 기사 {info.get('found', 0)}건을 수집했지만 입력과의 유사도 기준"
                      f"({ASK_CONFIG['related_min_score']})을 넘는 구절이 없습니다.")
            if info.get("failed"):
                reason += f" (처리 실패 기사 {len(info['failed'])}건)"
        else:
            reason = "수집한 근거 기사만으로는 질문에 직접 답할 수 없다고 모델이 판단했습니다."

    return {
        "question": rag.get("question"),
        "answer": rag.get("answer", ""),
        "answer_lines": [] if insufficient else _answer_lines(rag.get("answer", "")),
        "insufficient": insufficient,
        "insufficient_reason": reason,
        "sources": sources,
        "source_count": len(sources),
        "chunk_count": sum(len(s.get("chunk_ids", [])) for s in rag.get("sources", [])),
        "keywords": info.get("keywords", []),
        "queries": info.get("queries", []),
        "related_found": info.get("found", 0),
        "validation": {"ok": bool(val.get("auto_checks_ok", True)), "errors": val.get("errors", []),
                       "checked_lines": val.get("checked_lines", 0)},
        "generation_ms": gen.get("generation_time_ms") if gen else None,
        "total_ms": round(elapsed_ms, 1),
        "model": gen.get("model") if gen else None,
        "truncated": bool(gen.get("truncated")) if gen else False,
        "plain_answer": plain.get("answer") if plain else None,
    }


# ──────────────── FastAPI ────────────────
app = FastAPI()


@app.get("/", response_class=HTMLResponse)
def index():
    return HTML


@app.exception_handler(Exception)
async def all_errors(request, exc):
    traceback.print_exc()
    return JSONResponse(status_code=500, content={"detail": f"{type(exc).__name__}: {exc}"})


@app.post("/api/ask")
def ask(body: AskInput):
    question = " ".join((body.question or "").split())
    if not question:
        raise HTTPException(status_code=400, detail="내용을 입력해주세요.")
    if len(question) > MAX_QUESTION_CHARS:
        raise HTTPException(status_code=400, detail=f"{MAX_QUESTION_CHARS:,}자 이내로 입력해주세요.")
    if len(jobs) > 100:                                   # 오래된 작업 기록 정리
        for old_id in list(jobs)[:50]:
            jobs.pop(old_id, None)
    job_id = uuid.uuid4().hex[:12]
    jobs[job_id] = {"status": "queued", "percent": 0, "stage": "대기 중", "error": None, "result": None}
    _job_queue.put((job_id, question, body.compare))
    return {"job_id": job_id}


@app.get("/api/ask/status/{job_id}")
def ask_status(job_id: str):
    job = jobs.get(job_id)
    if job is None:
        raise HTTPException(status_code=404, detail="작업을 찾을 수 없습니다.")
    return job


class EvalBatchInput(BaseModel):
    items: list[dict]            # [{"question": "...", "reference": "(선택) 정답 문장"}]


@app.post("/api/eval/batch")
def eval_batch(body: EvalBatchInput):
    items = [it for it in body.items if " ".join(str(it.get("question", "")).split())]
    if not items:
        raise HTTPException(status_code=400, detail="질문을 한 개 이상 입력해주세요.")
    if len(items) > MAX_EVAL_BATCH:
        raise HTTPException(status_code=400, detail=f"한 번에 {MAX_EVAL_BATCH}개까지 평가할 수 있습니다.")
    if any(len(str(it.get("question", ""))) > MAX_QUESTION_CHARS for it in items):
        raise HTTPException(status_code=400, detail=f"질문은 {MAX_QUESTION_CHARS:,}자 이내로 입력해주세요.")
    job_id = uuid.uuid4().hex[:12]
    jobs[job_id] = {"status": "queued", "percent": 0, "stage": "대기 중", "error": None, "result": None}
    _job_queue.put((job_id, {"items": items}, True))
    return {"job_id": job_id}


@app.get("/api/eval/summary")
def eval_summary():
    """지금까지 UI에서 처리한 모든 질문의 누적 평균 지표."""
    return eval_log.summary()



@app.get("/api/eval/csv/{job_id}")
def eval_csv(job_id: str):
    path = f"eval_batch_{job_id}.csv"
    if not re.fullmatch(r"[0-9a-f]{12}", job_id) or not os.path.isfile(path):
        raise HTTPException(status_code=404, detail="CSV 파일을 찾을 수 없습니다.")
    return FileResponse(path, filename=path, media_type="text/csv")


# ──────────────── 서버 실행 및 ngrok 터널링 ────────────────
_prev = globals().get("server")
if _prev is not None:
    _prev.should_exit = True
    time.sleep(1)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()

ngrok.set_auth_token(userdata.get("ITF_TOKENS"))        # 반드시 본인토큰으로 적어서 사용해주세요.
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)
tunnel = ngrok.connect(8000)
print("접속 주소:", tunnel.public_url)
print("서버 실행 중입니다. 이 셀은 계속 실행 상태로 두세요. 종료하려면 셀의 ■ 중지 버튼을 누르세요.")

# GPU 작업(임베딩·vLLM 생성)은 모델을 적재한 메인 스레드에서 처리합니다.
# vLLM이 프로세스 안에서 실행 중이라(멀티프로세싱 꺼짐) 다른 스레드에서 호출하면 CUDA 오류가 날 수 있습니다.
try:
    _worker(_job_queue)
except KeyboardInterrupt:
    pass
finally:
    server.should_exit = True
    try:
        ngrok.disconnect(tunnel.public_url)
    except Exception:
        pass
    print("서버를 종료했습니다.")

## **중복 토큰에러시 실행해주세요**
# 1. 2. 3 순으로 실행해주세요

In [ ]:
# CELL 1
!rm /root/.config/ngrok/ngrok.yml

In [ ]:
# CELL 2
!rm ~/.config/ngrok/ngrok.yml

In [ ]:
# CELL 3
!ngrok config add-authtoken ITF_TOKENS